# 第 14 章　本地 AI 应用前置技能

> **本章定位**：第 15 章两个「本地 AI 实战案例」要调用本机 Ollama 大模型，中间隔着一层**工程技能**：装环境、找文件、读图片、发请求、解析 JSON、清洗文本、落盘校验。本章专门补上这一层，让学生看得懂、改得动、验得了，而不是照抄一段黑盒代码。
>
> **知识点范围**：`venv` / `pip` / `requirements.txt`；`pathlib` 与 `glob`；`json` 与 JSON Schema 字段校验；`base64` 与 MIME；`urllib.request` 本地 HTTP POST；`re` 正则；`csv` 与 `collections.Counter`；Ollama 本地服务与 RTX 3050 显存档位。
>
> **环境准备**：
> - 本章全部代码**只用 Python 标准库**，`pip install` 一个包都不用装；
> - 本章全部示例**不联网、不依赖 Ollama、不依赖显卡**，在任何一台装了 Python 3 的机器上输出都一致，可当堂逐字校对；
> - 真实调用本机大模型从 **14.9 部署完成**后开始，在第 15 章正式使用。
>
> **本章约定**：允许使用第 1—13 章全部语法；所有文件路径都用 `pathlib` 处理，不再手工拼字符串；所有模型返回值都按「先解析、再校验、后使用」三步走，对应第 9 章的容错思想。
>
> **编号约定**：本章「场景导入」单独占 14.0，不占知识点编号；知识点号与节号一致。第 1—13 章与 5A 节采用「知识点号 = 节号 − 1」，跨章引用一律以「知识点 X.Y」为准。

> **先修知识**：第 01—13 章全部语法；本机已安装 Python 3.8 及以上；不要求预装任何第三方库。第 15 章的 `--live` 调用还依赖本章 14.9 的部署结果。
>
> **学习目标**：完成本章后，你能够：
>
> 1. 用 `venv` 建虚拟环境并导出 `requirements.txt`，说明激活前后 `pip install` 的安装位置差异；
> 2. 用 `pathlib` 遍历指定目录，按扩展名筛选并打印文件名与文件大小（保留 1 位小数）；
> 3. 用 `json.loads` 与 `json.dumps` 完成 1 次序列化往返，指出 `ensure_ascii=False` 前后的输出差异；
> 4. 用 `re` 从 5 条事件文本中抽取车牌与时间，写出正则并逐段解释含义；
> 5. 用 `urllib.request` 向本机地址发起 1 次带 `timeout` 的 POST，服务未启动时捕获异常并打印降级提示。

---

## 本章学习路径

| 阶段 | 内容 | 编号 | 作用 |
|---|---|---|---|
| 第一部分 | 知识讲解 | 知识点 14.1 — 14.8 | 8 项工程技能，每项含原理 / 示例 / 变式 / 实操 / FAQ |
| 操作手册 | 本机部署全流程 | 14.9 | RTX 3050 + Ollama 从零部署，含 4GB / 8GB 双档 |
| 第二部分 | 代码示范 | 案例 14-1、案例 14-2 | 图片目录清点 / 服务请求与结构化返回校验 |
| 第三部分 | 布置练习 | 练习 14-1、练习 14-2 | 事件文本清洗 / 标注结果校验（各含 2 处口径问题） |
| 第四部分 | 习题答案 | 答案 14-1、答案 14-2 | 完整参考代码与对照解析 |

**知识点与代码对照表**：

| 知识点 | 名称 | 讲解位置 | 代码示范 | 对应练习 |
|---|---|---|---|---|
| 14.1 | 虚拟环境与包管理 | 14.1 | — | — |
| 14.2 | `pathlib` 路径与目录遍历 | 14.2 | 案例 14-1 | 练习 14-1 |
| 14.3 | `json` 序列化与字段校验 | 14.3 | 案例 14-2 | 练习 14-2 |
| 14.4 | `base64` 图片编码与 MIME | 14.4 | 案例 14-1 | — |
| 14.5 | `urllib.request` 本地 HTTP | 14.5 | 案例 14-2 | — |
| 14.6 | `re` 正则文本抽取 | 14.6 | — | 练习 14-1 |
| 14.7 | `csv` 与 `Counter` 统计 | 14.7 | — | 练习 14-1 |
| 14.8 | Ollama 服务与显存档位 | 14.8 | 案例 14-2 | 练习 14-2 |

## 教学提示（教师用）

- **建议课时**：10 学时（5 次课）。其中 14.1—14.7 约 6 学时，14.8—14.9 部署实操约 2 学时，案例与练习约 2 学时。
- **讲解主线**：按第 15 章一次真实调用的**时间顺序**倒推——先有环境（14.1）→ 找到图片（14.2）→ 图片转码（14.4）→ 发请求（14.5）→ 收 JSON（14.3）→ 清洗文本（14.6）→ 统计落盘（14.7）→ 服务从哪来（14.8）。建议开课先画这条链路，学生才知道每个库「为什么现在学」。
- **学生常见卡点**：把 `Path` 当字符串用 `+` 拼接；`json.dumps` 忘记 `ensure_ascii=False` 导致中文变成 `\uXXXX`；把 base64 的 `bytes` 直接写进 JSON；`urlopen` 不加 `timeout` 卡死；把模型返回的**字符串**当**字典**直接用；`re` 里 `\d` 写成普通字符串被转义；用「记录条数」当「去重后的类别数」。
- **课堂活动建议**：14.9 部署环节分组记录 `nvidia-smi` 显存、`ollama list` 模型清单与首次推理耗时，填一张「本机配置卡」。这张卡第 15 章要直接用来选模型，做到「上一章的产出是下一章的输入」。
- **安全与边界**：Ollama 默认只监听 `127.0.0.1:11434`，不对外提供服务；模型权重**首次 `ollama pull` 需要联网下载**，之后断网即可运行；全程无账号、无 API Key、无外部在线推理服务。必须向学生讲清「离线运行 ≠ 下载阶段也离线」。

---

# 第一部分　知识讲解

## 14.0 场景导入：一次本地 AI 调用到底经过几步

以「让本机模型判读一张路口抓拍图，输出拥堵等级」为例，完整链路是：

| 步骤 | 动作 | 用到的知识 | 本章位置 |
|---|---|---|---|
| 1 | 建虚拟环境、装依赖 | `venv` / `pip` | 14.1 |
| 2 | 找到 `images/` 下所有待标注图片 | `pathlib` / `glob` | 14.2 |
| 3 | 读成 `bytes`，转 base64，查 MIME | `base64` | 14.4 |
| 4 | 组 JSON 请求体，POST 给本机 11434 端口 | `json` + `urllib.request` | 14.3 / 14.5 |
| 5 | 收到响应，取 `message.content` | `json` | 14.3 |
| 6 | 校验字段是否齐全、取值是否合法 | 字段校验 + `try/except` | 14.3 / 14.9 |
| 7 | 清洗事件文本、统计词频 | `re` / `Counter` | 14.6 / 14.7 |
| 8 | 结果写成 JSON / CSV 落盘 | `csv` / `json` | 14.7 |

前 13 章解决的是「把数据算对」，本章解决的是「让数据进出程序」。第 15 章的两个案例，本质上就是这条链路跑两遍：一遍传图片，一遍传文本。

---

## 14.1 知识点 14.1　虚拟环境与包管理

### 14.1.1 核心概念

**为什么需要虚拟环境**：一台电脑上可能同时开着好几个 Python 项目，A 项目要 `pygame==2.1`，B 项目要 `pygame==2.6`，装到同一个全局解释器里就会互相覆盖。虚拟环境（`venv`）本质是**一份解释器的轻副本 + 一个独立的 `site-packages` 目录**：激活后，`python` 与 `pip` 都指向这个副本，装什么包都不影响别的项目。

**三个角色的分工**：

| 名称 | 角色 | 说明 |
|---|---|---|
| `venv` | 建环境 | `python3 -m venv .venv` 生成 `.venv/` 目录 |
| `pip` | 装包 | `python3 -m pip install 包名` 从 PyPI 下载并放入当前环境 |
| `requirements.txt` | 记清单 | 逐行写「包名 + 版本约束」，别人一条命令即可复现同一套依赖 |

**原理补充（解释器视角）**：激活虚拟环境只是把 `.venv/bin` 插到 `PATH` 最前面，并设置一个 `VIRTUAL_ENV` 环境变量；Python 启动时按 `sys.prefix` 找标准库与第三方包，虚拟环境里 `sys.prefix` 指向 `.venv`，因此 `import` 的搜索路径与全局隔离。判断当前是否在虚拟环境里，常用 `sys.prefix != sys.base_prefix`。这也是**推荐写 `python3 -m pip` 而不是直接写 `pip`** 的原因：前者明确绑定到当前解释器，不会出现「装到了另一个 Python 上」的经典故障。

### 14.1.2 语法要点

```bash
python3 -m venv .venv                 # 建环境（只需一次）
source .venv/bin/activate            # macOS / Linux 激活
.venv\Scripts\activate               # Windows 激活
python3 -m pip install pygame>=2.1   # 装包（注意版本符要加引号时写 "pygame>=2.1"）
python3 -m pip freeze > requirements.txt   # 导出当前环境的精确版本
python3 -m pip install -r requirements.txt # 按清单复现
deactivate                           # 退出环境
```

- `requirements.txt` 每行一条：`pygame>=2.1`（最低版本）、`ollama==0.4.7`（锁死版本）、`Pillow<=11.0`（最高版本）；`#` 开头为注释，空行忽略；
- 国内网络可加镜像：`-i https://pypi.tuna.tsinghua.edu.cn/simple`；
- `.venv/` **不入库、不打包**，靠 `requirements.txt` 复现，这一点要写进 `.gitignore`。

### 14.1.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 直接敲 `pip` | 包装到了系统 Python，项目里 `import` 不到 | 统一 `python3 -m pip` |
| 忘记激活 | 终端提示符没有 `(.venv)`，装包全局生效 | 每次新开终端先 `source` |
| 版本符未加引号 | `>` 被 shell 当成重定向，生成奇怪文件 | 写 `"pygame>=2.1"` |
| 把 `.venv/` 提交到仓库 | 体积上百 MB，且换台机器不能用 | `.gitignore` 加 `.venv/` |
| `pip freeze` 导出全部包 | 清单里混进无关依赖 | 只保留本项目真正用到的 |

### 14.1.4 随堂示例：解析 `requirements.txt`

**核心要点**：不真的读文件，用一段字符串模拟清单内容，练「跳过注释与空行 → 拆出包名 / 约束符 / 版本号」。这是后面自动检查依赖是否齐全的基础。

In [ ]:
requirements_text = """# 本地 AI 实战依赖
pygame>=2.1
ollama==0.4.7
Pillow>=10.0
"""

packages = []
for line in requirements_text.splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    for op in (">=", "==", "<="):
        if op in line:
            name, version = line.split(op, 1)
            packages.append((name.strip(), op, version.strip()))
            break

print("有效依赖行数：", len(packages))
for name, op, version in packages:
    print(name, "约束", op, version)

**预期输出**：

```text
有效依赖行数： 3
pygame 约束 >= 2.1
ollama 约束 == 0.4.7
Pillow 约束 >= 10.0
```

**变式代码（迁移练习）**：同一份清单里出现重复依赖时，按包名去重并报告重复项。用第 5 章的「先建空列表再追加」配合 `in` 判断即可。

In [ ]:
text = "# 依赖\npygame>=2.1\n\nollama==0.4.7\n# 重复项\npygame>=2.1\n"
seen = []
for line in text.splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    name = line.split(">=")[0].split("==")[0].strip()
    if name in seen:
        print("重复依赖：", name)
    else:
        seen.append(name)
print("去重后依赖：", seen)
print("条数：", len(seen))

**预期输出**：

```text
重复依赖： pygame
去重后依赖： ['pygame', 'ollama']
条数： 2
```

### 14.1.5 实操步骤（上机跟随）

1. 在课程目录打开终端，执行 `python3 -m venv .venv`，观察目录里多出 `.venv/bin`、`lib`、`pyvenv.cfg`。
2. 执行 `source .venv/bin/activate`，确认提示符左侧出现 `(.venv)`。
3. 执行 `python3 -c "import sys; print(sys.prefix != sys.base_prefix)"`，应看到 `True`；若 `False` 说明没激活成功。
4. 执行 `python3 -m pip list`，记录当前包数量（新建环境通常只有 `pip`、`setuptools` 两三个）。
5. 打开项目根目录的 `requirements.txt`，对照 14.1.4 的解析逻辑，说出每一行属于 `>=` / `==` / 注释中的哪一类。
6. 执行 `deactivate`，再执行第 3 步，应看到 `False`。

### 14.1.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `source: .venv/bin/activate: No such file` | 用的是 Windows，或建环境时路径写错 | Windows 改 `.venv\Scripts\activate`；先确认 `.venv` 已生成 |
| 激活后 `pip install` 仍装到全局 | 用了绝对路径的 `pip`，或 shell 别名干扰 | 一律 `python3 -m pip install ...` |
| `pygame>=2.1` 执行后目录下多了个名为 `=2.1` 的文件 | `>` 被 shell 解释为输出重定向 | 版本符加引号：`"pygame>=2.1"` |
| `pip install` 卡在下载 | 默认源在境外 | 追加 `-i https://pypi.tuna.tsinghua.edu.cn/simple` |
| 换台电脑照着 `requirements.txt` 装完仍报错 | 清单锁的是包版本，Python 版本没锁 | 记录 `python3 --version`，全组统一同一个大版本 |

---

## 14.2 知识点 14.2　`pathlib` 路径与目录遍历

### 14.2.1 核心概念

**为什么不用字符串拼路径**：Windows 用 `\`，macOS / Linux 用 `/`；`"images" + "/" + name` 在 Windows 上会出错，且容易漏斜杠、多斜杠。`pathlib.Path` 把路径当成**对象**，用 `/` 运算符拼层级，由 Python 按当前操作系统选分隔符。

`pathlib` 有两类对象，本章都会用到：

| 类 | 是否访问磁盘 | 用途 |
|---|---|---|
| `PurePath` / `PurePosixPath` / `PureWindowsPath` | 否 | 纯路径计算：取名字、后缀、父目录、拼接 |
| `Path` | 是 | 真实 IO：`exists()`、`glob()`、`read_bytes()`、`mkdir()` |

**原理补充（对象视角）**：`Path` 内部存的不是一整串字符，而是分成「锚点（盘符 / 根）+ 各级部件」的序列；`p.parts` 可以看到这个元组。`/` 运算符被 `__truediv__` 重载，右操作数会作为新部件追加，因此 `p / "labels" / "a.json"` 与字符串拼接有本质区别：它知道哪里该加分隔符、遇到绝对路径右操作数时会替换而非追加。

### 14.2.2 语法要点

| 写法 | 作用 | 返回 |
|---|---|---|
| `Path("images")` | 构造相对路径对象 | `Path` |
| `p / "cam01.jpg"` | 拼一层 | `Path` |
| `p.name` / `p.stem` / `p.suffix` | 文件名 / 主干名 / 扩展名 | `str` |
| `p.parent` | 父目录 | `Path` |
| `p.exists()` / `p.is_file()` / `p.is_dir()` | 判断存在与类型 | `bool` |
| `p.glob("*.jpg")` | 匹配当前层文件（生成器） | 可迭代 |
| `p.rglob("*.jpg")` | 递归匹配所有子层 | 可迭代 |
| `p.read_bytes()` / `p.read_text(encoding="utf-8")` | 读二进制 / 文本 | `bytes` / `str` |
| `p.mkdir(parents=True, exist_ok=True)` | 建目录（连带父目录、已存在不报错） | `Path` |
| `p.write_text(...)` / `p.write_bytes(...)` | 写文件 | `int` |

- `glob` 返回**生成器**，要反复用就转 `list()`；要稳定顺序就套 `sorted()`；
- 后缀比较前先 `.lower()`，否则 `CAM01.JPG` 会被漏掉；
- `stem` 去掉**最后一层**后缀，`cam01.tar.gz` 的 `stem` 是 `cam01.tar`。

### 14.2.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| `p + "/labels"` | `TypeError`，`Path` 不支持 `+` | 改 `p / "labels"` |
| 把 `Path` 传进只接受字符串的老接口 | 类型报错 | `str(p)` 显式转换 |
| `glob` 结果直接用两次 | 第二次为空（生成器已耗尽） | `files = sorted(p.glob("*.jpg"))` |
| 后缀大小写 | `.JPG` 图片被跳过 | `p.suffix.lower()` |
| 没判断 `is_file()` | 把同名子目录也当图片读 | 遍历时先 `if not p.is_file(): continue` |
| `mkdir()` 重复执行 | `FileExistsError` | 加 `exist_ok=True` |

### 14.2.4 随堂示例：路径对象的拆解与拼接

**核心要点**：`PurePosixPath` 不碰磁盘，适合当堂演示「取名字 / 取后缀 / 拼出同名标注文件」；再做一次 Windows 路径转 POSIX，说明跨平台差异。

In [ ]:
from pathlib import PurePosixPath, PureWindowsPath

p = PurePosixPath("/data/traffic/images/cam01_0800.jpg")
print("文件名：", p.name)
print("主干名：", p.stem)
print("扩展名：", p.suffix)
print("父目录：", p.parent)
print("上级目录名：", p.parent.parent.name)
print("拼接标注路径：", p.parent / "labels" / (p.stem + ".json"))

w = PureWindowsPath(r"D:\traffic\images\cam01.jpg")
print("Windows 路径转 POSIX：", w.as_posix())

**预期输出**：

```text
文件名： cam01_0800.jpg
主干名： cam01_0800
扩展名： .jpg
父目录： /data/traffic/images
上级目录名： traffic
拼接标注路径： /data/traffic/images/labels/cam01_0800.json
Windows 路径转 POSIX： D:/traffic/images/cam01.jpg
```

**变式代码（迁移练习）**：换成真实 `Path`，用 `tempfile.TemporaryDirectory()` 造一个临时图片目录再遍历。这样示例在任何机器上都能跑，又完整练到 `glob` / `is_file` / `mkdir`。

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "images").mkdir()
    for name in ("cam01.jpg", "cam02.jpg", "cam03.png", "notes.txt"):
        (root / "images" / name).write_text("x", encoding="utf-8")
    pics = sorted(p.name for p in (root / "images").glob("*.jpg"))
    print("jpg 文件：", pics)
    print("数量：", len(pics))
    print("labels 目录是否存在：", (root / "labels").exists())
    (root / "labels").mkdir(exist_ok=True)
    print("建目录后：", (root / "labels").exists())

**预期输出**：

```text
jpg 文件： ['cam01.jpg', 'cam02.jpg']
数量： 2
labels 目录是否存在： False
建目录后： True
```

> `with TemporaryDirectory() as tmp` 退出时会自动删除整个临时目录，因此反复运行不会在电脑上留垃圾文件——这是课堂演示文件 IO 最安全的写法（对应第 8 章 `with` 管理资源、第 9 章异常兜底）。

### 14.2.5 实操步骤（上机跟随）

1. 在 Jupyter 新建单元，写 `from pathlib import Path`，把 `Path("images/cam01.jpg")` 的 `name / stem / suffix / parent` 各打印一次，与 14.2.4 的输出逐项对照。
2. 把后缀改成 `.JPG`，观察 `suffix` 原样返回大写；再加 `.lower()`，观察归一化结果。
3. 复制 14.2.4 的变式代码，把 `glob("*.jpg")` 改成 `glob("*")`，预测会打印哪些名字，再运行验证（注意应包含 `cam03.png` 与 `notes.txt`）。
4. 把 `glob` 改成 `rglob`，在第 3 步的目录里再建一层 `images/sub/` 并放一张 `cam04.jpg`，观察递归效果。
5. 删掉 `exist_ok=True` 后连续运行两次，记录报错类型；再加回来确认不再报错。

### 14.2.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `TypeError: unsupported operand type(s) for +` | 把 `Path` 当字符串拼 | 一律用 `/` |
| `glob` 第二次遍历为空 | 生成器只能消费一次 | 转 `list()` 或 `sorted()` 存起来 |
| 图片清单顺序每次不同 | 文件系统返回顺序不保证 | `sorted(...)` |
| 中文路径报编码错 | 终端 / 编辑器编码不一致 | 统一 UTF-8，读写文本显式 `encoding="utf-8"` |
| `read_bytes()` 抛 `IsADirectoryError` | 把子目录也当文件 | 遍历时先 `p.is_file()` 过滤 |
| 相对路径在 Jupyter 里找不到文件 | Notebook 的工作目录是**课件所在文件夹** | 用 `Path.cwd()` 先打印确认，必要时用 `Path("..") / "images"` |

---

## 14.3 知识点 14.3　`json` 序列化与字段校验

### 14.3.1 核心概念

**JSON 是什么**：一种与语言无关的**文本**数据格式，只允许对象（对应 Python `dict`）、数组（`list`）、字符串、数字、`true/false`、`null` 六类值。Ollama 的请求体、响应体、模型吐出的结构化标注结果，全都是 JSON 文本。

| Python | JSON | 转换方向 |
|---|---|---|
| `dict` | 对象 `{}` | `dumps` / `loads` |
| `list` / `tuple` | 数组 `[]` | `dumps` / `loads` |
| `True` / `False` | `true` / `false` | `dumps` / `loads` |
| `None` | `null` | `dumps` / `loads` |
| `str` | 字符串 | `dumps` / `loads` |

四个函数别混：`dumps`/`loads` 处理**字符串**（`s` = string），`dump`/`load` 处理**文件对象**。

**原理补充（编码视角）**：`json.dumps` 默认 `ensure_ascii=True`，会把每个中文字符转成 `\uXXXX` 六字符转义序列。数据本身没错，但人看不懂、日志难查，且请求体变长。加 `ensure_ascii=False` 后输出真正的 UTF-8 中文；发 HTTP 请求时还要 `.encode("utf-8")` 变成 `bytes`，因为网络只传字节不传字符串。反过来，`json.loads` 既接受 `str` 也接受 `bytes`，所以 `json.loads(resp.read())` 不用先 `decode`。

**关键的工程观念**：`format` 里的 JSON Schema 属于**语法约束**（grammar-constrained decoding），只能保证模型返回的东西「能被 `json.loads` 解析」，**不能保证字段取值合理**。模型完全可能返回合法的 `{"level": "严重拥堵"}`。因此必须有第二步**语义校验**：检查必需字段是否齐全、取值是否在枚举内、数值范围是否合理。这正是第 9 章 `try/except` 与练习 14-2 要练的东西。

### 14.3.2 语法要点

```python
import json
s = json.dumps(obj, ensure_ascii=False, indent=2)  # 字典 -> 字符串（中文不转义、缩进 2）
obj = json.loads(s)                                 # 字符串/bytes -> 字典
json.dump(obj, f, ensure_ascii=False)               # 写文件
obj = json.load(f)                                  # 读文件
```

- `indent=2` 只用于给人看，发请求时不要加（会多传空白字节）；
- 解析失败抛 `json.JSONDecodeError`，它是 `ValueError` 的子类，可用 `e.msg` 取简短原因；
- 字段校验三件套：`[k for k in required if k not in data]`（缺字段）、`data["x"] in LEVELS`（取值合法）、`isinstance(v, (int, float))`（类型正确）；
- 模型返回的 `message.content` **常常还是一个 JSON 字符串**，需要 `loads` 两次：一次解响应，一次解内容。

### 14.3.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 忘 `ensure_ascii=False` | 中文变 `\u673a\u573a\u5317` | 所有 `dumps` 都补上 |
| 把 `dict` 直接塞进 `Request(data=...)` | `TypeError`，需要 bytes | `json.dumps(...).encode("utf-8")` |
| 把 `message.content` 当字典 | `TypeError: string indices must be integers` | 再 `json.loads(content)` 一次 |
| 只做 `try/except` 不做字段校验 | 能解析但缺字段，后面 `KeyError` | 解析 + 校验两步都做 |
| 用 `data.get("level", "拥堵")` 兜底 | 悄悄把坏数据改成好数据，统计失真 | 坏数据记入不合格，不伪造默认值 |
| `True` 写成 `"true"` | JSON 里变成字符串而非布尔 | 用 Python 的 `True` |

### 14.3.4 随堂示例：序列化、还原与中文

**核心要点**：观察 `True → true`、`None → null`，以及 `ensure_ascii=False` 对中文的影响。

In [ ]:
import json

record = {"路段": "机场北", "平均车速": 35.5, "拥堵": True, "车牌": None}
s = json.dumps(record, ensure_ascii=False)
print("序列化：", s)
print("类型：", type(s).__name__)
back = json.loads(s)
print("还原后：", back["路段"], back["平均车速"])
print("None 还原为：", back["车牌"])

**预期输出**：

```text
序列化： {"路段": "机场北", "平均车速": 35.5, "拥堵": true, "车牌": null}
类型： str
还原后： 机场北 35.5
None 还原为： None
```

**变式代码（迁移练习）**：模拟一条模型返回，做「必需字段 + 类型」校验，再故意解析一段截断的 JSON，看异常信息。

In [ ]:
import json

raw = '{"road": "机场北", "level": "拥堵", "avg_speed": 35.5}'
data = json.loads(raw)
required = ["road", "level", "avg_speed"]

missing = [k for k in required if k not in data]
print("缺失字段：", missing)
print("level 类型：", type(data["level"]).__name__)
print("校验通过：", not missing and isinstance(data["avg_speed"], (int, float)))

bad = '{"road": "机场北", "level": '
try:
    json.loads(bad)
except json.JSONDecodeError as e:
    print("解析失败：", e.msg)

**预期输出**：

```text
缺失字段： []
level 类型： str
校验通过： True
解析失败： Expecting value
```

### 14.3.5 实操步骤（上机跟随）

1. 运行 14.3.4 主示例，把 `ensure_ascii=False` 删掉再跑一次，对比中文是否变成 `\uXXXX`。
2. 在 `record` 里加一个列表 `["拥堵", "缓行"]`，观察 `dumps` 后 JSON 数组的形状。
3. 把 `required` 改成 `["road", "level", "avg_speed", "camera_id"]`，重新跑变式代码，确认「缺失字段」打印出 `['camera_id']`、「校验通过」变 `False`。
4. 把 `bad` 换成 `{'level': '拥堵'}`（单引号），观察 `json.loads` 报 `Expecting property name enclosed in double quotes`——JSON 只认双引号。
5. 用第 9 章知识给变式代码补一层「校验不通过时把样本编号写进 `invalid` 列表」，为练习 14-2 做准备。

### 14.3.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `json.decoder.JSONDecodeError` | 模型输出带解释文字、截断或单引号 | 先 `try/except`，再要求模型只输出 JSON；必要时重试一次 |
| `TypeError: Object of type Path is not JSON serializable` | 把 `Path` / `datetime` 等非 JSON 类型放进字典 | 先 `str(p)` 转成字符串 |
| `KeyError: 'level'` | 只解析没校验 | 用列表推导式先算 `missing` |
| 中文乱码 | 写文件时没指定编码 | `open(..., encoding="utf-8")` + `ensure_ascii=False` |
| 加了 `format` 仍返回非法取值 | 语法约束 ≠ 语义正确 | 必做枚举校验，见案例 14-2 |
| `loads` 后拿到的是 `str` 不是 `dict` | 模型把整个 JSON 又包了一层引号 | 打印 `type`，必要时二次 `loads` |

---

## 14.4 知识点 14.4　`base64` 图片编码与 MIME

### 14.4.1 核心概念

**为什么图片要转 base64**：HTTP 请求体是**文本**（JSON），而图片是任意字节的**二进制**，里面可能有 `\x00`、换行、非 UTF-8 字节，直接塞进 JSON 会破坏结构。base64 用 64 个可打印 ASCII 字符（`A-Z a-z 0-9 + /`，末尾用 `=` 补齐）重新表示任意字节，于是二进制就能安全地当成字符串传输。

**体积代价**：每 3 个字节 → 4 个 base64 字符，体积约膨胀 **1/3**（准确为 `4 × ceil(n/3)`）。所以大图要先缩放压缩再编码，这也是 14.4.6 FAQ 的高频问题。

**MIME 类型**：告诉接收方「这串字节是什么格式」。Ollama 的 `/api/chat` 接受 `images` 数组，元素就是**纯 base64 字符串**（不带 `data:image/...;base64,` 前缀）；而写网页内联图片时才需要 data URL 前缀。两者别混。

| 扩展名 | MIME |
|---|---|
| `.jpg` / `.jpeg` | `image/jpeg` |
| `.png` | `image/png` |
| `.webp` | `image/webp` |

**原理补充（位运算视角）**：base64 把字节流按 6 bit 一组切分（原本 8 bit 一组），每个 6 bit 值映射到一张 64 字符表。3 字节 = 24 bit = 4 组 6 bit，正好 4 个字符；不足 3 字节时用 0 补位并在末尾加 1—2 个 `=`。因此**编码是可逆的无损表示，不是压缩**，解码后字节完全一致（示例中的「还原一致： True」就是在验证这一点）。

### 14.4.2 语法要点

```python
import base64
b64_bytes = base64.b64encode(raw_bytes)     # bytes -> bytes
b64_str   = b64_bytes.decode("ascii")       # 转可放进 JSON 的 str
raw_bytes = base64.b64decode(b64_str)       # str/bytes -> 原始 bytes
```

- `b64encode` 的输入输出**都是 `bytes`**，要放 JSON 必须 `.decode("ascii")`；
- `b64decode` 可以直接吃 `str`，不必先编码回去；
- 常用扩展名到 MIME 的映射用字典 + `.get(后缀, "application/octet-stream")` 给兜底；
- Ollama 请求里的字段是 `"images": [b64_str]`，**列表**，即使只有一张图。

### 14.4.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 把 `b64encode` 的 `bytes` 直接塞进字典 | `json.dumps` 报不可序列化 | `.decode("ascii")` |
| 用 `read_text()` 读图片 | `UnicodeDecodeError` | 图片必须 `read_bytes()` |
| 加了 `data:image/...` 前缀发给 Ollama | 模型报图片格式错 | 只传纯 base64 载荷 |
| 后缀大写 `.JPG` 查不到 MIME | 走了兜底类型 | `.suffix.lower()` |
| 原图 5 MB 直接编码 | 请求体 6.7 MB，推理慢、显存吃紧 | 先缩放 / 转 JPEG 压缩 |
| 把 base64 当加密 | 误以为数据安全 | base64 只是编码，任何人可解码 |

### 14.4.4 随堂示例：编码、解码与 MIME 判定

**核心要点**：用一段伪造的 PNG 头字节模拟图片，观察 23 字节 → 32 字符（膨胀约 1/3），并验证解码后与原始字节完全一致。

In [ ]:
import base64
from pathlib import PurePosixPath

raw = b"\x89PNG\r\n\x1a\n" + b"FAKE-IMAGE-DATA"
encoded = base64.b64encode(raw).decode("ascii")
print("原始字节数：", len(raw))
print("base64 长度：", len(encoded))
print("前 16 字符：", encoded[:16])
print("还原一致：", base64.b64decode(encoded) == raw)

p = PurePosixPath("/img/cam01.png")
mime = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png"}.get(
    p.suffix.lower(), "application/octet-stream")
print("MIME：", mime)

**预期输出**：

```text
原始字节数： 23
base64 长度： 32
前 16 字符： iVBORw0KGgpGQUtF
还原一致： True
MIME： image/png
```

> `iVBORw0KGgo` 是 PNG 文件头 `\x89PNG\r\n\x1a\n` 的 base64 前缀，看到它就说明编码的是 PNG——这是排查「传错格式」最实用的一条经验。

**变式代码（迁移练习）**：拼一个网页用的 data URL，并把「头部」与「载荷」拆开，强化「发给 Ollama 只要载荷」这个区分。

In [ ]:
import base64

payload = base64.b64encode(b"IMG").decode("ascii")
data_url = "data:image/jpeg;base64," + payload
head, body = data_url.split(",", 1)
print("头部：", head)
print("载荷：", body)
print("载荷长度：", len(body))
print("是否纯 ASCII：", body.isascii())

**预期输出**：

```text
头部： data:image/jpeg;base64
载荷： SU1H
载荷长度： 4
是否纯 ASCII： True
```

### 14.4.5 实操步骤（上机跟随）

1. 运行 14.4.4 主示例，把 `b"FAKE-IMAGE-DATA"` 改长或改短，记录「原始字节数 / base64 长度」两组数，验证 `4 × ceil(n/3)` 规律。
2. 把 `.decode("ascii")` 删掉后执行 `json.dumps({"images": [base64.b64encode(raw)]})`，观察报错，再加回来确认通过。
3. 用 `Path` 读一张本机真实图片（`read_bytes()`），打印长度与其 base64 前 16 字符，判断是 PNG 还是 JPEG。
4. 把路径后缀改成 `.JPG`，不加 `.lower()` 时观察 MIME 是否落到兜底值，再加 `.lower()` 修正。
5. 用 `b64decode` 把编码结果还原并 `write_bytes()` 写到临时目录，与原文件大小做比较，确认无损。

### 14.4.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `UnicodeDecodeError` 读图片 | 用了 `read_text()` | 改 `read_bytes()` |
| `Object of type bytes is not JSON serializable` | 忘了 `.decode("ascii")` | 编码后立刻转 `str` |
| 模型报 `invalid image` | 传了 data URL 前缀或空字符串 | 只传纯载荷，并打印 `len(b64)` 确认非空 |
| 请求体特别大、推理很慢 | 原图未压缩 | 课程案例中先把长边缩到 512—768 px 再编码 |
| base64 里出现中文或换行 | 复制粘贴时混入了别的字符 | 用代码生成，不要手抄 |
| 担心「编码后别人看不懂」 | 误解了 base64 的作用 | 明确它是编码不是加密，敏感车牌图片仍需脱敏 |

---

## 14.5 知识点 14.5　`urllib.request` 本地 HTTP 调用

### 14.5.1 核心概念

**本机服务也是「网络程序」**：`ollama serve` 启动后，会在本机 `127.0.0.1:11434` 开一个 HTTP 服务。Python 要跟它说话，走的是标准 HTTP 协议，只是服务器在自己电脑上，数据包不经过外网。因此**不需要任何第三方库**，`urllib.request` 就够了。

一次 POST 的三要素：

| 要素 | 内容 |
|---|---|
| 地址 | `http://127.0.0.1:11434/api/chat` |
| 方法 | `POST`（有请求体）；`/api/version`、`/api/tags` 用 `GET` |
| 请求头 | `Content-Type: application/json` |
| 请求体 | `json.dumps(...).encode("utf-8")` 得到的 `bytes` |

`127.0.0.1` 是**回环地址**，只指向本机；`localhost` 通常等价。这与访问 `https://xxx.com/api` 有本质区别：没有域名解析、没有 TLS、没有外部账号与密钥。

**原理补充（Request 对象视角）**：`Request(url, data=..., headers=..., method=...)` 只**构造**请求，不发出去；真正发送发生在 `urlopen(req, timeout=...)`。只要传了 `data`，`get_method()` 就自动返回 `POST`。表头名在 `Request` 内部会被 `.capitalize()` 规范化，所以 `headers={"Content-Type": ...}` 存进去后用 `get_header("Content-type")` 取。把「构造」和「发送」拆开，正是本章示例能**完全离线运行**的原因——我们只构造、检查，用模拟响应练解析，不真的联网。

### 14.5.2 语法要点

```python
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError
import json

req = Request(BASE + "/api/chat", data=body,
              headers={"Content-Type": "application/json"}, method="POST")
with urlopen(req, timeout=60) as resp:
    result = json.loads(resp.read().decode("utf-8"))
content = result["message"]["content"]
```

- `timeout` **必写**：本地服务没启动时，不设超时可能长时间卡住；首次加载模型较慢，第 15 章建议设 60—120 秒；
- `urlopen` 返回的对象支持 `with`，用完自动关闭；
- 连不上抛 `URLError`（它是 `OSError` 子类）；HTTP 状态码非 2xx 抛 `HTTPError`（`URLError` 子类，要先捕获）；
- `stream: False` 表示一次性拿完整回答；若为 `True`，返回的是多行 NDJSON，需要逐行解析，本章不用；
- 捕获顺序：`except HTTPError` → `except URLError` → `except json.JSONDecodeError`。

### 14.5.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 忘写 `timeout` | 服务未启时程序假死 | 每次 `urlopen` 都带超时 |
| `data=` 传字符串 | `TypeError`，需要 bytes | `.encode("utf-8")` |
| 忘 `Content-Type` | 服务端按默认类型解析，可能 400 | 显式写 JSON 头 |
| 先捕 `URLError` 再捕 `HTTPError` | 404 被当成连接失败，错误信息失真 | 子类在前 |
| 把 `ollama serve` 关了还调用 | `Connection refused` | 先 `check_service()` 或看终端服务窗口 |
| 用 `localhost` 在某些环境解析到 IPv6 | 偶发连不上 | 统一写 `127.0.0.1` |

### 14.5.4 随堂示例：只构造请求，不发网络

**核心要点**：验证 `get_method()`、`full_url`、请求头、请求体字节数，以及 `json.loads(req.data)` 能读回自己组装的 JSON。全程零网络。

In [ ]:
import json
from urllib.request import Request

payload = {
    "model": "qwen2.5:3b",
    "messages": [{"role": "user", "content": "判读路况"}],
    "stream": False,
}
body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
req = Request(
    "http://127.0.0.1:11434/api/chat",
    data=body,
    headers={"Content-Type": "application/json"},
    method="POST",
)
print("方法：", req.get_method())
print("地址：", req.full_url)
print("Content-Type：", req.get_header("Content-type"))
print("请求体字节数：", len(req.data))
print("stream：", json.loads(req.data)["stream"])

**预期输出**：

```text
方法： POST
地址： http://127.0.0.1:11434/api/chat
Content-Type： application/json
请求体字节数： 99
stream： False
```

**变式代码（迁移练习）**：用一段写死的「模拟响应」练「两次 `loads` + 取值」，并演示服务未启动时该捕什么异常。

In [ ]:
import json
from urllib.error import URLError

mock = json.dumps({
    "message": {"role": "assistant", "content": '{"level": "拥堵"}'},
    "done": True,
}, ensure_ascii=False).encode("utf-8")

data = json.loads(mock.decode("utf-8"))
content = data["message"]["content"]
label = json.loads(content)
print("content 原文：", content)
print("解析出的等级：", label["level"])
print("done：", data["done"])

err = URLError("Connection refused")
print("异常类型：", type(err).__name__)
print("属于 OSError：", isinstance(err, OSError))

**预期输出**：

```text
content 原文： {"level": "拥堵"}
解析出的等级： 拥堵
done： True
异常类型： URLError
属于 OSError： True
```

### 14.5.5 实操步骤（上机跟随）

1. 运行 14.5.4 主示例，把 `method="POST"` 删掉，观察 `get_method()` 仍是 `POST`（因为有 `data`），理解「构造期不发送」。
2. 把 `data=body` 改成 `data=None`，观察方法变回 `GET`，说明请求体决定方法。
3. 把 `ensure_ascii=False` 去掉，对比「请求体字节数」变大多少，理解中文转义对体积的影响。
4. 在变式代码里把 `content` 换成 `'{"level": "拥堵"'`（截断），运行后观察 `json.loads` 抛错，再用 `try/except json.JSONDecodeError` 包住。
5. 完成 14.9 部署后，把案例 14-2 的 `check_service()` 取消注释调用一次，记录返回 `(True, 'HTTP 200')` 或 `(False, ...)`，并说出各自原因。

### 14.5.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `URLError: Connection refused` | `ollama serve` 没启动或端口被改 | 先启服务；`ollama list` 能返回即服务在 |
| 程序长时间无响应 | 没设 timeout，或首次加载模型很慢 | 加 `timeout=120`，首次耐心等一次 |
| `HTTPError: 404` | 接口路径写错（如 `/api/generate` 拼错） | 对照 `/api/chat`、`/api/tags`、`/api/version` |
| `HTTPError: 400` | 请求体字段名错、`images` 不是列表 | 打印 `req.data` 逐字段核对 |
| `json.loads` 报 Expecting value | 拿到的是空串或 HTML 错误页 | 先 `print(resp.status, resp.read()[:200])` |
| 担心数据外传 | 把本机服务误认为云服务 | 地址是 `127.0.0.1`，可用断网自检验证（见 14.9 第 7 步） |

---

## 14.6 知识点 14.6　`re` 正则文本抽取

### 14.6.1 核心概念

**正则是什么**：用一套「模式字符串」描述文本的形状，让程序从自由文本里把时间、车牌、车速这类**格式固定、内容可变**的片段抠出来。前面第 4 章的 `strip()` / `replace()` 只能处理固定字符，面对「08:12 京A12345 机场北 追尾 车速35km/h」这种半结构化日志就无能为力了。

| 记号 | 含义 | 例子 |
|---|---|---|
| `\d` | 一个数字 | `\d{5}` = 5 位数字 |
| `{n}` / `{n,}` | 重复 n 次 / 至少 n 次 | `[\u4e00-\u9fa5]{2,}` |
| `[...]` | 字符集合 | `[A-Z]` 大写字母 |
| `+` / `*` / `?` | 至少 1 次 / 0 次以上 / 可选 | `\d+\.?\d*` |
| `(...)` | 捕获分组 | `(\d+)\s*km/h` |
| `\s` | 空白字符 | `\s*` 允许有空格也可没有 |
| `\|` | 或 | `机场北\|机场南` |

**四个常用函数的分工**：

| 函数 | 从哪找 | 找几个 | 返回 |
|---|---|---|---|
| `re.match(p, s)` | **只能从头** | 1 | `Match` 或 `None` |
| `re.search(p, s)` | 全文任意位置 | 1 | `Match` 或 `None` |
| `re.findall(p, s)` | 全文 | 全部 | `list`（有分组时只返分组） |
| `re.sub(p, repl, s)` | 全文 | 全部替换 | 新字符串 |

**原理补充（原始字符串）**：`\d` 在普通 Python 字符串里会被当成转义序列（虽然后来不认识的转义会原样保留，但 `\n`、`\t` 就会真的变），因此正则一律写成 `r"\d+"` 原始字符串。写成 `r"..."` 后，反斜杠原封不动交给 `re` 引擎解释，这是「正则能跑对」的第一前提。`re.compile()` 把模式预编译成对象，同一模式要用很多次时更省事，也便于起个 `PLATE` 这样的语义化名字。

### 14.6.2 语法要点

```python
import re
PLATE = re.compile(r"[\u4e00-\u9fa5][A-Z]\d{5}")
PLATE.search(line)                     # 返回 Match；用 .group() 取内容
re.findall(r"\d{2}:\d{2}", text)       # 所有时间
re.sub(PLATE, "【车牌】", text)         # 脱敏替换
re.split(r"[,=\s]+", s)               # 按多种分隔符切
```

- `Match` 对象要先判空：`m = re.search(...); m.group() if m else ""`；
- `findall` 里**有捕获分组时只返回分组内容**，示例 `(\d+)\s*km/h` 返回 `['35', '45']` 而不是带单位的整串；
- `[\u4e00-\u9fa5]` 是常用汉字区间，用来匹配「京」「机」等；车牌省份简称适用，但不覆盖全部生僻汉字；
- 脱敏是数据处理的基本职业习惯：车牌、人名进报表前先 `re.sub` 掉。

### 14.6.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 模式没写 `r` 前缀 | `\b`、`\n` 被 Python 先吃掉 | 一律 `r"..."` |
| 用 `match` 找中间的内容 | 永远返回 `None` | 换 `search` / `findall` |
| `search` 不判空直接 `.group()` | `AttributeError: 'NoneType'` | 先判 `if m` |
| `findall` 结果带分组 | 拿到 `['35']` 而非 `['35km/h']` | 想取整串就用非捕获 `(?:...)` |
| `\d+` 匹配到小数前半截 | `35.5` 只取 `35` | 用 `\d+\.?\d*` |
| 正则写得太宽 | 把不需要的文本也抠进来 | 加边界、用具体字符集合 |

### 14.6.4 随堂示例：从交通事件文本里抠字段

**核心要点**：一条 `for` 都不用写，三个 `findall` 分别取时间、车牌、车速，再 `sub` 做车牌脱敏。

In [ ]:
import re

text = "08:12 京A12345 在 机场北 发生追尾，车速 35km/h；08:15 京B67890 缓行 45km/h"
plates = re.findall(r"[\u4e00-\u9fa5][A-Z]\d{5}", text)
speeds = re.findall(r"(\d+)\s*km/h", text)
times = re.findall(r"\d{2}:\d{2}", text)
print("时间：", times)
print("车牌：", plates)
print("车速：", speeds)
print("脱敏后：", re.sub(r"[\u4e00-\u9fa5][A-Z]\d{5}", "【车牌】", text))

**预期输出**：

```text
时间： ['08:12', '08:15']
车牌： ['京A12345', '京B67890']
车速： ['35', '45']
脱敏后： 08:12 【车牌】 在 机场北 发生追尾，车速 35km/h；08:15 【车牌】 缓行 45km/h
```

**变式代码（迁移练习）**：把 `match` / `search` / `findall` / 分组 / `split` 放在同一段里对照，一次看清差别。

In [ ]:
import re

s = "路段=机场北, 平均车速=35.5"
print("match 从头匹配：", bool(re.match(r"\d+", s)))
print("search 全局找：", re.search(r"\d+\.?\d*", s).group())
print("findall 全部：", re.findall(r"\d+\.?\d*", s))
m = re.search(r"路段=(\S+?),\s*平均车速=([\d.]+)", s)
print("分组1：", m.group(1), "分组2：", m.group(2))
print("split 清洗：", re.split(r"[,=\s]+", s))

**预期输出**：

```text
match 从头匹配： False
search 全局找： 35.5
findall 全部： ['35.5']
分组1： 机场北 分组2： 35.5
split 清洗： ['路段', '机场北', '平均车速', '35.5']
```

### 14.6.5 实操步骤（上机跟随）

1. 运行 14.6.4 主示例，把车速改成 `35.5km/h`，观察 `(\d+)\s*km/h` 是否还能完整匹配（应只剩 `5`），体会边界问题。
2. 把模式换成 `(\d+\.?\d*)\s*km/h` 再跑，确认能取到 `35.5`。
3. 把 `re.match(r"\d+", s)` 换成 `re.match(r"路段", s)`，在变式代码里应看到 `True`，理解「从头」的含义。
4. 在 `text` 里再加一条 `京C99999`，预测 `plates` 长度，运行验证。
5. 用 `re.sub(r"\d{2}:\d{2}", "【时间】", text)` 把时间也脱敏，输出与第 4 步对照。

### 14.6.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 模式明明对却匹配不到 | 忘写 `r` 前缀，或全角/半角标点不一致 | 统一 `r"..."`，检查中文逗号 `，` |
| `'NoneType' object has no attribute 'group'` | `search` 没匹配到 | `m.group() if m else ""` |
| `findall` 少返回了单位部分 | 用了捕获分组 | 改非捕获 `(?:...)` 或不加括号 |
| 车牌漏掉新能源 6 位牌 | 模式写死 5 位 | 改 `\d{5,6}`，或在业务里明确只处理普通牌 |
| 正则越来越长看不懂 | 想一条表达式解决所有字段 | 拆成多个 `compile` 对象，各管一个字段 |
| 处理上千条日志很慢 | 循环里反复编译同一模式 | 顶层 `re.compile()` 一次 |

---

## 14.7 知识点 14.7　`csv` 与 `Counter` 统计

### 14.7.1 核心概念

**为什么用 `csv` 模块**：第 8 章用 `split(",")` 手工解析 CSV，遇到「字段里本身有逗号」「带引号的中文」「不同系统换行符不同」就会出错。`csv.DictReader` 按表头把每行变成字典，`csv.DictWriter` 反过来把字典写回去，转义与换行都由标准库处理。

**`Counter` 是什么**：`collections.Counter` 是 `dict` 的子类，专门为「计数」设计。给它一个可迭代对象就自动统计频次，还提供 `most_common(n)` 取前 n 名，`update()` 累加。第 5 章手写 `d[k] = d.get(k, 0) + 1` 的活，一行就够。

| 写法 | 作用 |
|---|---|
| `Counter(可迭代对象)` | 统计每个元素出现次数 |
| `c[k] += 1` | 手动累加（键不存在时默认为 0） |
| `c.most_common(2)` | 返回 `[('拥堵', 3), ('畅通', 1)]` |
| `sum(c.values())` | 总命中次数 |
| `csv.DictReader(f)` | 按表头读成字典序列 |
| `csv.DictWriter(f, fieldnames=[...])` | 按指定列写，`writeheader()` + `writerows()` |

**原理补充（为什么不手工 split）**：CSV 的正式规则里，含分隔符的字段会用双引号包住，字段内的双引号再写成两个；`split(",")` 会把一个字段拆成两半。`DictReader` 内部用状态机逐字符解析，能正确处理这些情况。另外 `DictWriter` 默认用 `\r\n` 换行，在 Windows 上若不用 `newline=""` 打开文件会多出空行；本章演示统一写 `lineterminator="\n"`，并用 `io.StringIO` 在内存里完成，避免留下临时文件。

### 14.7.2 语法要点

```python
import csv, io
from collections import Counter

rows = list(csv.DictReader(io.StringIO(raw_text)))   # 内存字符串当文件读
out = io.StringIO()
w = csv.DictWriter(out, fieldnames=["路段", "等级"], lineterminator="\n")
w.writeheader(); w.writerows(rows)
```

- `DictReader` 的字段名来自**第一行**，`list(rows[0].keys())` 可查看；
- `DictWriter` 的 `fieldnames` 必须显式给，否则不知道列顺序；写入的字典**不能有多余键**（会抛 `ValueError`），案例里常先挑字段构造新字典；
- 写真实文件：`with open(p, "w", newline="", encoding="utf-8-sig") as f:`，`utf-8-sig` 让 Excel 打开中文不乱码；
- `Counter` 保持**首次出现顺序**，相同计数按插入顺序排，因此 `dict(c)` 的键顺序可预测，方便逐字校对输出。

### 14.7.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 用 `split(",")` 解析 CSV | 中文字段含逗号时错列 | 用 `DictReader` |
| `DictWriter` 忘 `writeheader()` | 第一行变成数据，没有表头 | 先写表头 |
| 写入字典有多余键 | `ValueError: dict contains fields not in fieldnames` | 先挑出需要的列 |
| Windows 里每行间多空行 | `\r\n` 与文本模式叠加 | `newline=""` + `lineterminator="\n"` |
| Excel 打开中文乱码 | 文件是纯 UTF-8 无 BOM | `encoding="utf-8-sig"` |
| 把 `most_common()` 结果当字典 | 拿到的是 `list[tuple]` | 需要字典就 `dict(c.most_common())` |

### 14.7.4 随堂示例：读 CSV、统计、回写

**核心要点**：`io.StringIO` 把字符串伪装成文件对象，因此示例不产生任何磁盘文件，输出完全稳定。

In [ ]:
import csv
import io
from collections import Counter

raw = "路段,等级\n机场北,拥堵\n京开,畅通\n机场北,拥堵\n机场南,拥堵\n"
rows = list(csv.DictReader(io.StringIO(raw)))
print("记录数：", len(rows))
print("字段名：", list(rows[0].keys()))
levels = Counter(r["等级"] for r in rows)
print("等级统计：", dict(levels))
print("最常见的：", levels.most_common(1))

out = io.StringIO()
w = csv.DictWriter(out, fieldnames=["路段", "等级"], lineterminator="\n")
w.writeheader()
w.writerows(rows)
print("回写行数：", len(out.getvalue().splitlines()))

**预期输出**：

```text
记录数： 4
字段名： ['路段', '等级']
等级统计： {'拥堵': 3, '畅通': 1}
最常见的： [('拥堵', 3)]
回写行数： 5
```

**变式代码（迁移练习）**：NLP 案例的前置动作——中文片段切分 + 关键词计数。注意「按片段数」和「按词频」是两件不同的事。

In [ ]:
import re
from collections import Counter

text = "机场北发生追尾，机场北拥堵；京开缓行，机场北恢复畅通"
words = re.findall(r"[\u4e00-\u9fa5]{2,}", text)
keys = ["追尾", "拥堵", "缓行", "畅通"]
counts = Counter({k: text.count(k) for k in keys})
print("中文片段数：", len(words))
print("关键词计数：", dict(counts))
print("出现过的：", [k for k in keys if counts[k] > 0])
print("总命中：", sum(counts.values()))

**预期输出**：

```text
中文片段数： 4
关键词计数： {'追尾': 1, '拥堵': 1, '缓行': 1, '畅通': 1}
出现过的： ['追尾', '拥堵', '缓行', '畅通']
总命中： 4
```

### 14.7.5 实操步骤（上机跟随）

1. 运行 14.7.4 主示例，在 `raw` 里加一行 `机场南,缓行`，预测「记录数 / 等级统计 / 回写行数」三项新值，再运行验证。
2. 把 `levels.most_common(1)` 改成 `most_common(2)`，观察返回列表结构。
3. 给 `DictWriter` 的 `fieldnames` 加一列 `"备注"`，运行观察是否报错；再用 `extrasaction="ignore"` 参数修正。
4. 在变式代码里把 `keys` 增加 `"机场北"`，预测计数（应为 3），运行验证。
5. 把 `io.StringIO()` 换成真实文件写入（`open(..., newline="", encoding="utf-8-sig")`），用 Excel 或文本编辑器打开确认中文正常。

### 14.7.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 读出来的键带 `\ufeff` | 文件有 BOM | `encoding="utf-8-sig"` 读 |
| 写入后每行之间有空行 | Windows 下重复换行 | `newline=""` + `lineterminator="\n"` |
| `Counter` 顺序与预期不同 | 按首次出现顺序 + 计数排序 | 需要固定顺序就 `sorted(...)` |
| 词频统计把「车速」也算成关键词 | 关键词表未限定 | 维护明确的 `keys` 白名单 |
| `DictReader` 返回空 | 字符串首行不是表头，或空文件 | 检查首行内容 |
| 统计口径把记录数当类别数 | 忘记去重 | 用 `set(...)` 或 `Counter` 的键数，见练习 14-1 |

---

## 14.8 知识点 14.8　Ollama 服务与 RTX 3050 显存档位

### 14.8.1 核心概念

**Ollama 是什么**：一个在本机跑大模型的**服务程序**。它把模型权重加载进内存 / 显存，并提供一套本地 HTTP 接口；Python 只负责发请求、收结果。常用命令：

| 命令 | 作用 |
|---|---|
| `ollama serve` | 启动本地服务（默认 `127.0.0.1:11434`） |
| `ollama pull 模型名` | 下载模型权重（**首次需要联网**） |
| `ollama list` | 查看已安装模型、体积 |
| `ollama ps` | 查看当前已加载、正在占用显存的模型 |
| `ollama run 模型名` | 命令行对话，用于快速验证 |

**三个接口**：`/api/chat`（对话，第 15 章主用）、`/api/tags`（等价 `ollama list`）、`/api/version`（连通性自检）。

**显存决定模型**：RTX 3050 有台式 8GB 与笔记本 4GB / 6GB 多个版本，同名显卡能跑的模型完全不同。量化后的模型权重需要整体装进显存，装不下就会降级到 CPU 共享内存，速度骤降。

| 用途 | 推荐模型 | 约体积 | 8GB | 4GB / 6GB |
|---|---|---|---|---|
| 图片标注（首选） | `gemma3:4b` | ≈ 3.0—3.3 GB | ✅ 宽裕 | ⚠ 勉强 / 不建议 |
| 图片标注（轻量兜底） | `moondream` | ≈ 1.7 GB | ✅ 很快 | ✅ 可用 |
| 图片标注（高质量可选） | `qwen2.5vl:3b` / `llava:7b` | ≈ 3.2 / 4.7 GB | ✅ | ❌ 超显存 |
| 自然语言处理 | `qwen2.5:3b` | ≈ 1.9 GB | ✅ | ✅ |

> **不要选 11B / 13B 视觉模型**：在 3050 上会直接超显存或退化到 CPU，一次推理几分钟，课堂无法完成。

**原理补充（量化是什么）**：原始权重多为 16 位浮点数，一个 7B 参数模型要 14 GB；Ollama 默认分发的是 4-bit 量化版（Q4），把权重压到约 1/4，才让 3B—4B 模型能进 8GB 显存。代价是精度略降——这也是第 15 章强调「模型输出必须做字段校验、用已知答案样本验证」的根本原因：**本地小模型是概率生成，不是确定性查表**。

### 14.8.2 语法要点

- 模型名带 tag：`qwen2.5:3b` 与 `qwen2.5:7b` 是两个不同模型，`ollama pull` 与代码里的 `model` 字段必须**逐字一致**；
- `ollama list` 输出是表格文本，第 8.2 节的 `split()` 解析方式见 demo14-8b；
- 首次推理会把权重从磁盘加载进显存，耗时明显长于后续调用，属正常现象；
- 不需要手动设 `CUDA_VISIBLE_DEVICES`，Ollama 会自动探测 NVIDIA 驱动；驱动不可用时静默退到 CPU，所以要先用 `nvidia-smi` 确认；
- 断网后仍可推理，因为权重已在本地；但**首次 `pull` 必须联网**，这句话要写在实验报告里。

### 14.8.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 把 `4GB` 当 `8GB` 选模型 | 显存溢出、推理极慢 | 先 `nvidia-smi` 看实际容量 |
| 代码里的模型名漏了 `:3b` | `model not found` | 与 `ollama list` 逐字核对 |
| `ollama serve` 窗口被关 | `Connection refused` | 保持终端窗口开着，或设为后台服务 |
| 端口 11434 被占用 | 启动失败 | `lsof -i :11434` 查占用进程 |
| 以为断网也能 `pull` | 下载报错 | 把下载与运行分成两个阶段 |
| 期望每次输出一模一样 | 小模型有随机性 | 用结构化字段 + 已知答案样本验收，别逐字比对 |

### 14.8.4 随堂示例：按显存自动选模型

**核心要点**：把「查显存 → 选档位 → 定模型」写成函数，纯数据运算，不需要显卡就能当堂验证。第 15 章会复用同样的分支逻辑。

In [ ]:
VRAM_PROFILES = {
    8: {"vision": "gemma3:4b", "text": "qwen2.5:3b", "fallback": "moondream"},
    4: {"vision": "moondream", "text": "qwen2.5:3b", "fallback": "granite3.2-vision:2b"},
}

def recommend(vram_gb):
    key = 8 if vram_gb >= 8 else 4
    return key, VRAM_PROFILES[key]

for vram in (4, 6, 8):
    key, cfg = recommend(vram)
    print(vram, "GB → 采用", key, "GB 档 | 视觉", cfg["vision"], "| 文本", cfg["text"])

**预期输出**：

```text
4 GB → 采用 4 GB 档 | 视觉 moondream | 文本 qwen2.5:3b
6 GB → 采用 4 GB 档 | 视觉 moondream | 文本 qwen2.5:3b
8 GB → 采用 8 GB 档 | 视觉 gemma3:4b | 文本 qwen2.5:3b
```

**变式代码（迁移练习）**：解析 `ollama list` 的表格输出，算出还需要 `pull` 哪些模型。

In [ ]:
ollama_list = """NAME                ID              SIZE      MODIFIED
qwen2.5:3b          f9a1...         1.9 GB    2 hours ago
moondream:latest    c3b2...         1.7 GB    3 days ago
"""
installed = []
for line in ollama_list.splitlines()[1:]:
    if line.strip():
        installed.append(line.split()[0].split(":")[0])
print("已安装模型：", installed)
need = ["qwen2.5", "moondream", "gemma3"]
print("缺失模型：", [m for m in need if m not in installed])
print("需执行 ollama pull 的个数：", len([m for m in need if m not in installed]))

**预期输出**：

```text
已安装模型： ['qwen2.5', 'moondream']
缺失模型： ['gemma3']
需执行 ollama pull 的个数： 1
```

### 14.8.5 实操步骤（上机跟随）

1. 按 14.9 完成部署，记录 `nvidia-smi` 显示的显存总量，填进「本机配置卡」。
2. 执行 `ollama list`，把输出贴进 demo14-8b 的 `ollama_list` 变量，运行后核对「缺失模型」是否与实际一致。
3. 把本机显存值代入 `recommend(vram_gb)`，记录选定视觉模型与文本模型，写在配置卡上。
4. 执行 `ollama ps`，观察首次推理前后「已加载模型」的变化。
5. 断网后重跑一次 `check_service()` 与一次 `ollama run`，确认服务仍可用，把结果写进实验报告。

### 14.8.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `nvidia-smi` 命令不存在 | 驱动未装或非 NVIDIA 卡 | 先装对应驱动；纯 CPU 也能跑，但需明确告知学生速度差异 |
| `model not found, try pulling it first` | 模型名不一致或没下载 | `ollama list` 核对，再 `ollama pull` |
| 推理慢到几分钟 | 显存不足退到 CPU，或选了 7B+ 模型 | 换 `moondream` / `qwen2.5:3b` |
| `CUDA error: out of memory` | 模型与其他程序抢显存 | 关掉占显存的程序，或换更小模型 |
| 端口 11434 连不上 | 服务未启 / 被防火墙拦 / 端口占用 | `ollama serve` 保持运行；`lsof -i :11434` |
| 同一提示词两次回答不同 | 生成式模型的采样随机性 | 验收看结构化字段而非逐字文本；必要时设 `options.temperature=0` |

---

## 14.9 本机部署全流程（从零开始 · 上机跟随）

> 本节是**操作手册**，不计入知识点编号，但属于本章必做实验。用户当前状态为 **Ollama 尚未部署**，因此从驱动确认写起。请按顺序执行，每步都记录结果。

### 步骤 1　确认显卡与显存

```bash
nvidia-smi
```

看输出右上角的显存总量（如 `8192MiB` / `4096MiB`）。把数字填进下表：

| 项目 | 本机实测值 | 填写处 |
|---|---|---|
| 显卡型号 | 例：NVIDIA GeForce RTX 3050 | ______ |
| 显存总量 | 例：8192 MiB | ______ |
| 驱动版本 | 例：560.94 | ______ |
| 采用的档位 | 8GB 档 / 4GB 档 | ______ |

> 若 `nvidia-smi` 不存在，先到 NVIDIA 官网安装与操作系统匹配的驱动，重启后再执行。Ollama 在检测不到 GPU 时会退到 CPU，能跑但很慢；这种情况下建议直接选 `qwen2.5:3b` + `moondream`。

### 步骤 2　安装 Ollama

| 系统 | 做法 |
|---|---|
| macOS | 到 Ollama 官网下载 `.dmg`，拖入「应用程序」；首次打开后在菜单栏看到图标即服务已启动 |
| Windows | 官网下载安装程序，双击安装；安装完在终端执行 `ollama --version` 验证 |
| Linux | 官网提供一键安装脚本，或按发行版包管理器安装 |

验证：

```bash
ollama --version
```

### 步骤 3　启动本地服务

```bash
ollama serve
```

这个终端窗口**要保持打开**。看到监听 `127.0.0.1:11434` 的提示即成功。另开一个终端执行：

```bash
curl http://127.0.0.1:11434/api/version
```

正常应返回一段含版本号的 JSON。Python 侧等价写法见案例 14-2 的 `check_service()`。

### 步骤 4　按显存档位拉取模型（唯一需要联网的阶段）

**8GB 档（推荐配置）**：

```bash
ollama pull gemma3:4b        # 图片标注（视觉）
ollama pull qwen2.5:3b       # 自然语言处理（文本）
ollama pull moondream        # 轻量视觉兜底
```

**4GB / 6GB 档（降级配置）**：

```bash
ollama pull moondream        # 图片标注主用
ollama pull qwen2.5:3b       # 自然语言处理
```

> 下载耗时取决于网速，建议课前统一完成。下载完成后，后续所有环节都可以**断网运行**。

### 步骤 5　核对已安装模型

```bash
ollama list
ollama ps
```

`ollama list` 应能列出步骤 4 拉取的模型名与体积；把输出交给 demo14-8b 解析，算出还缺哪些。`ollama ps` 在尚未推理时通常为空。

### 步骤 6　命令行快速验证

```bash
ollama run qwen2.5:3b
```

输入「用一句话说明什么是拥堵」，看到中文回答即模型可正常工作。视觉模型可用 `ollama run moondream` 并附上图片路径验证。输入 `/bye` 退出。

### 步骤 7　断网自检（离线边界验证）

1. 断开网络（拔网线 / 关 Wi-Fi）；
2. 保持 `ollama serve` 运行；
3. 再次执行 `curl http://127.0.0.1:11434/api/version`，应仍返回版本号；
4. 再次 `ollama run qwen2.5:3b` 提问，应仍能回答。

这一步证明：**推理阶段不访问任何外部在线 API、不需要账号、不需要密钥**。请在实验报告中明确写出「下载阶段需联网、推理阶段可断网」，避免把「离线运行」误解成全程无需网络。

### 步骤 8　Python 侧连通性自检

完成以上步骤后，在项目目录执行案例 14-2，或在 Python 里调用：

```python
from case2_服务连通自检 import check_service
print(check_service())
```

期望看到 `(True, 'HTTP 200')`。若返回 `(False, '服务未启动或端口不通：...')`，回到步骤 3。

### 部署常见问题速查

| 现象 | 排查顺序 |
|---|---|
| `ollama: command not found` | 重装 → 确认 PATH → 重开终端 |
| `Connection refused` | `ollama serve` 是否在跑 → 端口 11434 是否被占 |
| 拉取到一半失败 | 网络不稳，重跑 `ollama pull`（支持断点续传） |
| 推理特别慢 | `ollama ps` 看是否退到 CPU → 换更小模型 |
| 显存不足报错 | 关掉浏览器 / 游戏等占显存程序 → 用 4GB 档模型 |
| 模型名对不上 | `ollama list` 复制完整名称（含 `:3b` 等 tag）到代码里 |

---

# 第二部分　代码示范

## 案例 14-1　待标注图片目录清点与 base64 预检

### 任务情境

标注组收到一批路口抓拍图，送入本地模型前要先确认几张可用、编码后体积多大；上传接口对 MIME 与体积都有限制，运维要求先出清单。没有真实图片时，用临时目录里 3 个文件把逻辑跑通，全程离线。

**任务**：给定一批路口抓拍图，遍历目录找出所有图片，判定 MIME、计算编码后 base64 长度，最后生成一份 JSON 清单。这是第 15 章「AI 图片标注」的第 2—3 步。

**涉及知识点**：14.2（`pathlib` / `glob`）、14.4（`base64` / MIME）、14.3（JSON 清单），以及第 5 章字典、第 6 章函数封装。

**说明**：为保证课堂可复现，示例用 `TemporaryDirectory` 临时造 3 张「伪图片」与 1 个干扰文件；实际使用时把 `root` 改为 `Path("images")` 即可，其余代码不用动。

### 输入规格

| 项 | 要求 |
|---|---|
| 目录 | `with TemporaryDirectory()` 建的临时目录；内写 4 个文件与 1 个子目录，不依赖真实图片 |
| 文件 | `cam01.jpg`（10 字节）、`cam02.png`（14 字节）、`cam03.jpg`（15 字节）、干扰文件 `notes.txt`、子目录 `labels`；统计 3 张 |
| 类型 | 遍历对象为 `Path`；字节数由 `read_bytes()` 得到 `bytes`；清单项为字典，值为 `str` 与 `int` |
| 取值范围 | 后缀限 `.jpg`、`.jpeg`、`.png`、`.webp`，转小写后查表；MIME 为 `image/jpeg` 或 `image/png` |
| 非法值 | 后缀大写 `CAM01.JPG`、无后缀文件、只有魔数的伪图片、目录不存在时 `glob` 抛 `FileNotFoundError` |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 终端恰好 6 行：1 行汇总 + 3 行明细 + 1 行 base64 总长 + 1 行清单长度 |
| 逐行格式 | 明细行以两个空格开头，字段用「空格 竖线 空格」分隔，依次为文件名、MIME、字节数、base64 长度；汇总行形如 `发现图片： 3 张（notes.txt 与 labels/ 已跳过）` |
| 数值 / 精度 | 整数无小数；base64 长度按 4 乘字节数除以 3 向上取整，10 字节对应 16，14 与 15 字节对应 20，合计 56 |
| 判定方式 | 与「运行结果」逐字比对 6 行；明细 3 行，文件名恰为 `cam01.jpg`、`cam02.png`、`cam03.jpg` |

In [ ]:
# -*- coding: utf-8 -*-
"""案例 14-1：待标注图片目录清点与 base64 预检（纯标准库，可离线运行）。

本例用 tempfile 临时造一批演示文件，避免依赖真实图片。
实际使用时把 root 改成 Path("images") 即可，其余代码不用动。
"""
import base64
import json
from pathlib import Path
from tempfile import TemporaryDirectory

MIME = {
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
    ".png": "image/png",
    ".webp": "image/webp",
}


def scan(root):
    """遍历目录，返回图片清单；非图片文件与子目录自动跳过。"""
    items = []
    for p in sorted(root.glob("*")):
        if not p.is_file():
            continue
        mime = MIME.get(p.suffix.lower())
        if mime is None:
            continue
        data = p.read_bytes()
        items.append({
            "file": p.name,
            "mime": mime,
            "bytes": len(data),
            "b64_len": len(base64.b64encode(data)),
        })
    return items


with TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "cam01.jpg").write_bytes(b"\xff\xd8" + b"IMG-01" + b"\xff\xd9")
    (root / "cam02.png").write_bytes(b"\x89PNG\r\n\x1a\n" + b"IMG-02")
    (root / "cam03.jpg").write_bytes(b"\xff\xd8" + b"IMG-03-LONG" + b"\xff\xd9")
    (root / "notes.txt").write_text("课堂记录", encoding="utf-8")
    (root / "labels").mkdir()

    items = scan(root)
    print("发现图片：", len(items), "张（notes.txt 与 labels/ 已跳过）")
    for it in items:
        print(" ", it["file"], "|", it["mime"], "|", it["bytes"], "字节 | base64", it["b64_len"], "字符")
    print("base64 总长度：", sum(it["b64_len"] for it in items), "字符")
    manifest = json.dumps({"count": len(items), "items": items}, ensure_ascii=False)
    print("清单 JSON 长度：", len(manifest), "字符")

### 运行结果

```text
发现图片： 3 张（notes.txt 与 labels/ 已跳过）
  cam01.jpg | image/jpeg | 10 字节 | base64 16 字符
  cam02.png | image/png | 14 字节 | base64 20 字符
  cam03.jpg | image/jpeg | 15 字节 | base64 20 字符
base64 总长度： 56 字符
清单 JSON 长度： 241 字符
```

### 解析

1. **`sorted(root.glob("*"))`**：先排序再遍历，保证同一批文件在任何电脑上输出顺序一致，方便逐字校对。
2. **两层过滤**：`is_file()` 排掉 `labels/` 子目录，`MIME.get(...)` 排掉 `notes.txt`。少了第一层，`read_bytes()` 会对目录抛 `IsADirectoryError`。
3. **为什么算 `b64_len` 而不真的编码两次**：真实图片动辄几百 KB，课堂演示只需知道「编码后有多大」即可估算请求体体积；本例 `read_bytes()` 后顺手算一次，避免重复编码。
4. **`ensure_ascii=False`**：清单里若出现中文路段名，不加这个参数会变成 `\uXXXX`，清单难以人工核对。
5. **与第 15 章的接口**：真实案例里把 `base64.b64encode(data).decode("ascii")` 的结果放进 `"images"` 列表，本例的 `b64_len` 就是那条字符串的长度。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 把 `notes.txt` 也统计进去 | 只判断了 `is_file()`，没查后缀 | 用 MIME 字典 `.get()` 过滤 |
| 后缀大写图片被跳过 | 没 `.lower()` | `p.suffix.lower()` |
| `labels/` 触发 `IsADirectoryError` | 没判断 `is_file()` | 遍历开头 `continue` |
| 每次运行输出顺序不同 | `glob` 不保证顺序 | 套 `sorted()` |
| 临时文件残留 | 直接写进项目目录 | 用 `with TemporaryDirectory()` |

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| 明细出现 4 行，含 `notes.txt` | 失败：后缀过滤缺失 | 查 `MIME.get(...)` 为 `None` 时是否 `continue` |
| 报错 `IsADirectoryError` | 失败：没有跳过子目录 | 查是否有 `if not p.is_file(): continue` |
| 三次运行的文件顺序不同 | 失败：`glob` 未排序 | 查是否写 `sorted(root.glob("*"))` |
| `cam02.png` 的 MIME 为 `image/jpeg` | 失败：字典键值写错 | 核对 `MIME` 中 `.png` 一行 |
| base64 总长不是 56 | 失败：字节数或编码方式不符 | 核对三个文件的字节数与 b64 长度 |
| 清单 JSON 长度不是 241 | 失败：字段名、顺序或 `ensure_ascii` 被改 | 核对键序 file、mime、bytes、b64_len |

---

## 案例 14-2　本地服务请求构造与结构化返回校验

### 任务情境

服务端同事发来了 `/api/chat` 的字段约定，但本机还没装好模型。你要先按 `format` 的 JSON Schema 拼好请求体，核对方法与字节数，再用三条模拟返回走通「解析、必填字段、枚举取值」三层校验，全程断网。

**任务**：构造一条发往本机 Ollama `/api/chat` 的 POST 请求（含 `format` JSON Schema），再对三条模拟返回做「解析 → 必填字段 → 枚举取值」三层校验。

**涉及知识点**：14.5（`urllib.request`）、14.3（`json` / 字段校验）、14.8（Ollama 接口），以及第 9 章异常兜底。

**说明**：主程序**不发起真实网络请求**，输出在任何机器上一致；`check_service()` 是部署完成后才调用的真实自检函数，用法见 14.9 步骤 8。

### 输入规格

| 项 | 要求 |
|---|---|
| 数据来源 | 模型名 `qwen2.5:3b` 与提示语 `判读：机场北平均车速 35.5km/h` 为字面量；三条模拟返回写在 `samples` |
| 类型 | 请求体由 `json.dumps(...).encode("utf-8")` 得到 `bytes`；`samples` 三个元素均为 `str` |
| 请求体字段 | 顶层 `model`、`messages`、`stream`、`format`；`messages` 首条含 `role` 与 `content`；`format` 含 `type`、`properties`、`required` |
| 取值 | `stream` 为假；`format.type` 为 `object`；`required` 为 `level` 与 `road`；地址固定 `http://127.0.0.1:11434` |
| 非法值 | `data=` 直接传字典、`format` 写成字符串、样本 3 截断成不合法 JSON、`urlopen` 不带 `timeout` |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 终端恰好 9 行：4 行请求信息 + 1 行分隔 + 3 行样本 + 1 行提示 |
| 逐行格式 | 前 4 行为「字段名： 值」；分隔行为 `—— 结构化返回校验 ——`；样本行为「样本 N ： 结论 竖线 数据或 None」 |
| JSON 键 | 请求体键序 `model`、`messages`、`stream`、`format`；校验通过的记录键序 `road`、`level` |
| 精度 / 单位 | `请求体字节数` 为整数，应为 260；`format.required` 打印为 Python 列表，元素用单引号 |
| 判定方式 | 与「运行结果」逐字比对 9 行；核对第 1 行 `POST` 与第 2 行地址；样本 2 须有 `缺字段：level`，样本 3 须有 `JSON 解析失败 →` |

In [ ]:
# -*- coding: utf-8 -*-
"""案例 14-2：本地 Ollama 服务的请求构造与结构化返回解析（离线演示版）。

主程序不发起真实网络请求，输出在任何机器上一致，便于课堂统一校对；
check_service() 供本机部署完成后调用，步骤见讲义 14.8。
"""
import json
from urllib.error import URLError
from urllib.request import Request, urlopen

BASE = "http://127.0.0.1:11434"


def build_request(model, prompt):
    """构造 /api/chat 的 POST 请求，并用 format 约定返回结构。"""
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "format": {
            "type": "object",
            "properties": {
                "level": {"type": "string"},
                "road": {"type": "string"},
            },
            "required": ["level", "road"],
        },
    }
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    return Request(BASE + "/api/chat", data=body,
                   headers={"Content-Type": "application/json"}, method="POST")


def check_service(timeout=2):
    """真实连通性自检：返回 (是否连通, 说明)。本机部署后再调用。"""
    try:
        with urlopen(BASE + "/api/version", timeout=timeout) as resp:
            return True, "HTTP %d" % resp.status
    except URLError as e:
        return False, "服务未启动或端口不通：%s" % e.reason
    except Exception as e:
        return False, "未知错误：%s" % e


def parse_label(content):
    """把模型返回的 content 解析成标注字典，并做字段与取值校验。"""
    data = json.loads(content)
    missing = [k for k in ("level", "road") if k not in data]
    if missing:
        return None, "缺字段：" + "、".join(missing)
    if data["level"] not in ("畅通", "缓行", "拥堵"):
        return None, "level 取值非法：" + str(data["level"])
    return data, "校验通过"


req = build_request("qwen2.5:3b", "判读：机场北平均车速 35.5km/h")
print("请求方法：", req.get_method())
print("请求地址：", req.full_url)
print("请求体字节数：", len(req.data))
print("format.required：", json.loads(req.data)["format"]["required"])

# 模拟三条模型返回：一条正常、一条缺字段、一条不是合法 JSON
samples = [
    '{"road": "机场北", "level": "拥堵"}',
    '{"road": "京开"}',
    '{"road": "机场南", "level": "拥堵",',
]
print("—— 结构化返回校验 ——")
for i, content in enumerate(samples, 1):
    try:
        label, msg = parse_label(content)
        print("样本", i, "：", msg, "|", label)
    except json.JSONDecodeError as e:
        print("样本", i, "：JSON 解析失败 →", e.msg)
print("提示：连通性自检函数 check_service() 需本机部署 Ollama 后调用")

### 运行结果

```text
请求方法： POST
请求地址： http://127.0.0.1:11434/api/chat
请求体字节数： 260
format.required： ['level', 'road']
—— 结构化返回校验 ——
样本 1 ： 校验通过 | {'road': '机场北', 'level': '拥堵'}
样本 2 ： 缺字段：level | None
样本 3 ：JSON 解析失败 → Expecting property name enclosed in double quotes
提示：连通性自检函数 check_service() 需本机部署 Ollama 后调用
```

### 解析

1. **`build_request()` 只构造、不发送**：`Request(...)` 生成对象后可直接检查 `get_method()` 与 `req.data`，所以本例断网也能跑。真正发送要 `urlopen(req, timeout=...)`，放在 `check_service()` 之外的第 15 章。
2. **`format` 是语法约束**：`required: ["level", "road"]` 让模型倾向于返回可解析的 JSON，但**不保证取值合法**。样本 1 合法，样本 2 缺字段，样本 3 直接截断——三种情况都必须能处理。
3. **`parse_label()` 返回二元组**：`(数据, 说明)`，校验失败时返回 `(None, 原因)`。调用方一眼能区分「没数据」和「数据为空」，不用靠异常传递业务信息。
4. **异常分层**：`json.JSONDecodeError` 在调用处捕获（解析层），字段与取值问题在 `parse_label` 内判定（业务层）。这正是第 9 章「只捕自己能处理的异常」的应用。
5. **`check_service()` 的兜底顺序**：`URLError` 覆盖连不上，`Exception` 覆盖其他意外；返回 `(bool, str)` 而不是直接 `print`，便于第 15 章在主程序里据此决定是否降级。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| `data=` 传字典 | `TypeError` | `json.dumps(...).encode("utf-8")` |
| 把 `content` 当字典用 | 少了一次 `loads` | 先 `json.loads(content)` |
| 以为加了 `format` 就不用校验 | 语法正确 ≠ 语义正确 | 必做枚举与范围校验 |
| `urlopen` 无 timeout | 服务未启时卡住 | 每次都写 `timeout` |
| 校验失败悄悄填默认值 | 统计结果失真 | 记入不合格样本，见练习 14-2 |

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| 第 1 行为「请求方法： GET」 | 失败：`Request` 未指定方法 | 查 `method="POST"` 参数 |
| 报错 `TypeError` | 失败：`data` 传了字典 | 查是否 `json.dumps(...).encode("utf-8")` |
| 样本 1 被判「缺字段」 | 失败：`content` 未先解析 | 查 `parse_label()` 首行是否 `json.loads` |
| 样本 2 被判「校验通过」 | 失败：没有做必填校验 | 查 `missing` 是否覆盖 `level` 与 `road` |
| 样本 3 没有报 `JSON 解析失败` | 失败：`try / except` 位置错 | 查是否捕获 `json.JSONDecodeError` |
| 请求体字节数不是 260 | 失败：提示语或模型名被改 | 核对 `qwen2.5:3b` 与提示语两个字面量 |

---

# 第三部分　布置练习

## 练习 14-1　交通事件文本清洗与词频统计

- **对应知识点**：14.6（`re` 正则文本抽取）、14.7（`csv` 与 `Counter` 统计）
- **提交文件**：`practice1_事件文本清洗.py`（UTF-8，附一次运行截图，含修正前后输出）
- **难度**：基础　**建议用时**：20 分钟

### 任务情境

上游要把路口巡查的 5 条自由文本整理成表，`clean()` 的抽取逻辑已跑通，但报表里「涉及路段数」直接用了记录条数 5，「低速车辆」的阈值误写成小于等于 35，把刚好 35 km/h 的 `京A12345` 也算了进去。你只改这两处，让路段去重为 3、低速车剩 2 辆，其余输出与数据别动。全程离线，不碰显卡。

### 题目

`practice1_事件文本清洗.py` 已经能运行，但**有 2 处统计口径问题**。请对照题目要求修正，使输出与第四部分答案 14-1 一致。

### 要求

1. 「涉及路段数」应统计**去重后的路段种类数**（5 条记录里只涉及 3 个路段），不能直接用记录条数；
2. 「低速车辆」的口径是**严格小于 35 km/h**（`< 35`），边界值 35 km/h 不计入；
3. 保持前 5 行输出（记录数、首条、CSV 行数、关键词计数、命中总数）完全不变；
4. 只改这 2 处，不改动 `clean()` 函数与 `EVENTS` 数据。

### 提示

第 1 处用第 5 章的 `set` 去重；第 2 处是第 3 章的边界条件，可回看 05A 节练过的「宽条件在前」类 bug。

### 自检要点（不含答案）

- [ ] 「涉及路段数」输出为 `3`，判定方法：对 `rows` 的「路段」字段做 `set` 后取长度；
- [ ] 「低速车辆（<35）」输出为 `['京B67890', '京D33344']`，共 2 辆，判定方法：逐条核对 35 km/h 的 `京A12345` 不在列表内；
- [ ] 前 5 行输出与「运行结果（当前代码，尚未修正）」逐字一致；
- [ ] `clean()` 与 `EVENTS` 未被修改，判定方法：与题目给出的代码块逐行比对。

In [ ]:
# -*- coding: utf-8 -*-
"""练习 14-1：交通事件文本清洗与词频统计。

当前代码可以运行，但有 2 处统计口径问题，请按《案例与练习.md》练习 14-1
的题目要求修正，使输出与第四部分参考答案一致。
"""
import csv
import io
import re
from collections import Counter

EVENTS = [
    "08:12 京A12345 机场北 追尾 车速35km/h",
    "08:15 京B67890 机场北 拥堵 车速30km/h",
    "08:21 京C11122 京开 缓行 车速48km/h",
    "08:33 京D33344 机场南 拥堵 车速25km/h",
    "08:40 京E55566 机场北 畅通 车速62km/h",
]
KEYWORDS = ["追尾", "拥堵", "缓行", "畅通"]
PLATE = re.compile(r"[\u4e00-\u9fa5][A-Z]\d{5}")
ROAD = re.compile(r"机场北|机场南|京开|京港澳")


def clean(lines):
    """把每条自由文本拆成结构化字典。"""
    rows = []
    for line in lines:
        plate = PLATE.search(line)
        time = re.search(r"\d{2}:\d{2}", line)
        speed = re.search(r"(\d+)\s*km/h", line)
        road = ROAD.search(line)
        rows.append({
            "时间": time.group() if time else "",
            "车牌": plate.group() if plate else "",
            "路段": road.group() if road else "",
            "车速": int(speed.group(1)) if speed else 0,
        })
    return rows


rows = clean(EVENTS)
print("清洗记录数：", len(rows))
print("首条：", rows[0]["时间"], rows[0]["车牌"], rows[0]["路段"], rows[0]["车速"])

out = io.StringIO()
w = csv.DictWriter(out, fieldnames=["时间", "车牌", "路段", "车速"], lineterminator="\n")
w.writeheader()
w.writerows(rows)
print("CSV 行数（含表头）：", len(out.getvalue().splitlines()))

counts = Counter()
for line in EVENTS:
    for k in KEYWORDS:
        if k in line:
            counts[k] += 1
print("关键词计数：", dict(counts))
print("关键词命中总数：", sum(counts.values()))

# 口径问题 1：直接拿记录条数当「涉及路段数」，没有按路段去重
print("涉及路段数：", len(rows))

# 口径问题 2：低速阈值写成 <= 35，题目要求是 < 35
slow = [r["车牌"] for r in rows if r["车速"] <= 35]
print("低速车辆（<35）：", slow)

### 运行结果（当前代码，尚未修正）

```text
清洗记录数： 5
首条： 08:12 京A12345 机场北 35
CSV 行数（含表头）： 6
关键词计数： {'追尾': 1, '拥堵': 2, '缓行': 1, '畅通': 1}
关键词命中总数： 5
涉及路段数： 5
低速车辆（<35）： ['京A12345', '京B67890', '京D33344']
```

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 路段去重 | 4 | 用 `set` 或 `Counter` 的键数，结果为 3 |
| 边界条件 | 3 | 阈值改为 `< 35`，低速车 2 辆 |
| 前 5 行保持不变 | 2 | 未误改 `clean()` 与关键词统计 |
| 代码可读性 | 1 | 有注释说明两处修正 |

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 修正两处口径后运行 `practice1_事件文本清洗.py` | 共 7 行；第 6 行 `涉及路段数（去重）： 3`，第 7 行 `低速车辆（<35）： ['京B67890', '京D33344']` | 与答案 14-1「运行结果」7 行逐字比对 |
| T2 | 边界 | 首条记录由 `车速35km/h` 改为 `车速34km/h` | 低速变 3 辆且含 `京A12345`；「首条：」车速变 34 | 核对「低速车辆」与「首条：」两行 |
| T3 | 异常 | 把 `京开` 改成未收录的 `京通快速` | 去重路段数变 2，空串不参与计数 | 集合长度为 2，列表不含空串 |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| 路段去重 | 运行脚本并读第 6 行 | 输出 3，列出 `京开`、`机场北`、`机场南` | 4 |
| 边界条件 | 读「低速车辆（<35）：」行 | 恰为 `['京B67890', '京D33344']`，不含 `京A12345` | 3 |
| 前 5 行不变 | 与「运行结果」前 5 行逐字比对 | 5 行完全一致 | 2 |
| 未改数据与清洗函数 | 与题目代码块逐行比对 | 无任何差异 | 1 |
| 合计 | — | — | 10 |

---

## 练习 14-2　模型标注结果的解析、字段校验与落盘

- **对应知识点**：14.3（`json` 序列化与字段校验）、14.7（`csv` 与 `Counter` 统计）
- **提交文件**：`practice2_标注结果校验.py`（UTF-8，附一次运行截图，含修正前后输出）
- **难度**：进阶　**建议用时**：25 分钟

### 任务情境

质检小结里，5 条返回中样本 3 缺 `level`、样本 4 等级写成「严重拥堵」、样本 5 是截断的 JSON，代码却只 `continue`，两个数字都失真。你要把它们记进 `invalid`，分母换回 `len(RAW)`，再补 `valid` 的落盘。全程离线、不占显卡，最终应为 3 条不合格、40.0 % 与 2 条落盘。

### 题目

`practice2_标注结果校验.py` 已经能运行，但**有 2 处统计口径问题**，导致「不合格条数」与「合格率」明显失真。请修正，使输出与第四部分答案 14-2 一致。

### 要求

1. 三类不合格样本（`JSON 解析失败` / `缺字段` / `level 取值非法`）都要记入 `invalid` 列表，不能只 `continue`；
2. 合格率的分母是**样本总数 `len(RAW)`**，不是合格条数；
3. 补充把 `valid` 序列化为 JSON 字符串（模拟落盘），并打印落盘条数与合格记录摘要；
4. 5 条样本的诊断输出（样本 3/4/5 三行）保持不变。

### 提示

这是第 9 章「异常也要留下痕迹」与 14.3「解析 + 校验两步走」的综合练习。真实项目里，漏记不合格样本会让质量报表虚高，是 AI 数据标注流水线最常见的坑。

### 自检要点（不含答案）

- [ ] 「不合格条数」输出为 `3`，判定方法：样本 3、4、5 各被追加一次到 `invalid`；
- [ ] 「合格率」输出为 `40.0 %`，判定方法：`len(valid) / len(RAW) * 100`，即 2 / 5；
- [ ] 落盘条数为 `2`，且能回读为 2 条记录，判定方法：`json.loads(json.dumps(valid, ensure_ascii=False))`；
- [ ] 样本 3/4/5 三行诊断信息与题目给出的输出逐字一致。

In [ ]:
# -*- coding: utf-8 -*-
"""练习 14-2：模型标注结果的解析、字段校验与落盘。

当前代码可以运行，但有 2 处统计口径问题，请按《案例与练习.md》练习 14-2
的题目要求修正，使输出与第四部分参考答案一致。
"""
import json

RAW = [
    '{"road": "机场北", "level": "拥堵", "avg_speed": 35.5}',
    '{"road": "京开", "level": "畅通", "avg_speed": 60.0}',
    '{"road": "机场南", "avg_speed": 30.0}',
    '{"road": "机场北", "level": "严重拥堵", "avg_speed": 25.0}',
    '{"road": "机场北", "level": "缓行",',
]
REQUIRED = ["road", "level", "avg_speed"]
LEVELS = ["畅通", "缓行", "拥堵"]

valid, invalid = [], []
for i, text in enumerate(RAW, 1):
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        print("样本", i, "：JSON 解析失败，已跳过")
        # 口径问题 1：不合格样本没有记入 invalid
        continue
    missing = [k for k in REQUIRED if k not in data]
    if missing:
        print("样本", i, "：缺字段", missing)
        # 口径问题 1：不合格样本没有记入 invalid
        continue
    if data["level"] not in LEVELS:
        print("样本", i, "：level 非法", data["level"])
        # 口径问题 1：不合格样本没有记入 invalid
        continue
    valid.append(data)

print("合格条数：", len(valid))
print("不合格条数：", len(invalid))
# 口径问题 2：分母用了合格条数，而不是样本总数
print("合格率：", round(len(valid) / len(valid) * 100, 1), "%")

### 运行结果（当前代码，尚未修正）

```text
样本 3 ：缺字段 ['level']
样本 4 ：level 非法 严重拥堵
样本 5 ：JSON 解析失败，已跳过
合格条数： 2
不合格条数： 0
合格率： 100.0 %
```

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 三类样本都记入 invalid | 4 | 不合格条数为 3 |
| 合格率分母修正 | 3 | 2 / 5 = 40.0% |
| 落盘与摘要 | 2 | `json.dumps(valid, ensure_ascii=False)` + 回读条数 |
| 诊断输出不变 | 1 | 样本 3/4/5 三行原样 |

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 修正两处口径后运行 `practice2_标注结果校验.py` | 共 8 行；`不合格条数： 3`、`合格率： 40.0 %`、`落盘 JSON 条数： 2` | 与答案 14-2「运行结果」8 行逐字比对 |
| T2 | 边界 | 样本 4 的 `level` 由 `严重拥堵` 改为 `拥堵` | 合格 3、不合格 2、合格率 60.0 %、落盘 3 条；诊断剩样本 3 与 5 | 核对 4 个数字，诊断 2 行 |
| T3 | 异常 | 删掉样本 2 的 `avg_speed` 键 | 诊断行 `样本 2 ：缺字段 ['avg_speed']`，不合格 4、合格率 20.0 %、落盘 1 | 诊断共 4 行，「不合格条数」为 4 |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| 三类样本都记入 | 读「不合格条数：」并数诊断行 | 输出 3，样本 3、4、5 各 1 次诊断 | 4 |
| 合格率分母 | 读「合格率：」行 | 输出 40.0 % | 3 |
| 落盘与回读 | 读「落盘 JSON 条数：」与「合格记录：」 | 2，且中文不转义 | 2 |
| 诊断输出不变 | 与题目输出前三行逐字比对 | 三行完全一致 | 1 |
| 合计 | — | — | 10 |

---

# 第四部分　习题答案

## 答案 14-1　交通事件文本清洗与词频统计（修正版）

### 参考代码（完整）

In [ ]:
# -*- coding: utf-8 -*-
"""练习 14-1 参考答案：交通事件文本清洗与词频统计（口径修正版）。

修正点：
1. 涉及路段数改为对「路段」字段去重后计数（set）；
2. 低速车辆阈值由 <= 35 改为 < 35，边界车速 35km/h 不计入。
"""
import csv
import io
import re
from collections import Counter

EVENTS = [
    "08:12 京A12345 机场北 追尾 车速35km/h",
    "08:15 京B67890 机场北 拥堵 车速30km/h",
    "08:21 京C11122 京开 缓行 车速48km/h",
    "08:33 京D33344 机场南 拥堵 车速25km/h",
    "08:40 京E55566 机场北 畅通 车速62km/h",
]
KEYWORDS = ["追尾", "拥堵", "缓行", "畅通"]
PLATE = re.compile(r"[\u4e00-\u9fa5][A-Z]\d{5}")
ROAD = re.compile(r"机场北|机场南|京开|京港澳")


def clean(lines):
    rows = []
    for line in lines:
        plate = PLATE.search(line)
        time = re.search(r"\d{2}:\d{2}", line)
        speed = re.search(r"(\d+)\s*km/h", line)
        road = ROAD.search(line)
        rows.append({
            "时间": time.group() if time else "",
            "车牌": plate.group() if plate else "",
            "路段": road.group() if road else "",
            "车速": int(speed.group(1)) if speed else 0,
        })
    return rows


rows = clean(EVENTS)
print("清洗记录数：", len(rows))
print("首条：", rows[0]["时间"], rows[0]["车牌"], rows[0]["路段"], rows[0]["车速"])

out = io.StringIO()
w = csv.DictWriter(out, fieldnames=["时间", "车牌", "路段", "车速"], lineterminator="\n")
w.writeheader()
w.writerows(rows)
print("CSV 行数（含表头）：", len(out.getvalue().splitlines()))

counts = Counter()
for line in EVENTS:
    for k in KEYWORDS:
        if k in line:
            counts[k] += 1
print("关键词计数：", dict(counts))
print("关键词命中总数：", sum(counts.values()))

# 修正 1：先取出去重的路段集合，再计数
roads = [r["路段"] for r in rows if r["路段"]]
print("涉及路段数（去重）：", len(set(roads)), sorted(set(roads)))

# 修正 2：严格小于 35
slow = [r["车牌"] for r in rows if r["车速"] < 35]
print("低速车辆（<35）：", slow)

### 运行结果

```text
清洗记录数： 5
首条： 08:12 京A12345 机场北 35
CSV 行数（含表头）： 6
关键词计数： {'追尾': 1, '拥堵': 2, '缓行': 1, '畅通': 1}
关键词命中总数： 5
涉及路段数（去重）： 3 ['京开', '机场北', '机场南']
低速车辆（<35）： ['京B67890', '京D33344']
```

### 解析

1. **修正 1**：`roads = [r["路段"] for r in rows if r["路段"]]` 先把路段字段取成列表，`len(set(roads))` 得到种类数 3。`sorted(set(roads))` 顺便打印路段清单，便于核对是不是漏了某条路。
2. **修正 2**：`<= 35` 改 `< 35`，把边界值 35 km/h 的 `京A12345` 排除，低速车从 3 辆变 2 辆。**「<」与「<=」在统计口径里是两道题**，这与 05A 节练习 5A-7 的边界 bug 同源。
3. **为什么用 `if r["路段"]` 过滤**：正则没匹配到时 `clean()` 填的是空字符串，空串不该算作一种「路段」。真实数据里这种脏值很常见。
4. **前 5 行为何不变**：`clean()`、`EVENTS`、`KEYWORDS` 都没动，只改统计口径，符合题目要求第 3 条。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 去重后仍是 5 | 对整个 `rows` 字典列表去重 | 先取出「路段」字段列 |
| 低速车还是 3 辆 | 只改了打印文字没改判断符号 | 检查 `if r["车速"] < 35` |
| 关键词计数变了 | 误改了 `KEYWORDS` 或循环 | 保持原样 |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 7 行输出，路段去重为 3、低速 2 辆 | 与「运行结果」一致 |
| T2 | 首条车速改 34 后低速变 3 辆，35 与 34 分居 `< 35` 两侧 | 验证符号是 `<` 而非 `<=` |
| T3 | `京通快速` 不被 `ROAD` 匹配，空串过滤后只余 2 种 | 见解析第 3 条 |

---

## 答案 14-2　模型标注结果的解析、字段校验与落盘（修正版）

### 参考代码（完整）

In [ ]:
# -*- coding: utf-8 -*-
"""练习 14-2 参考答案：模型标注结果的解析、字段校验与落盘（口径修正版）。

修正点：
1. 三类不合格样本（解析失败 / 缺字段 / 取值非法）统一记入 invalid；
2. 合格率分母改为样本总数 len(RAW)，并把合格记录序列化为 JSON 供落盘。
"""
import json

RAW = [
    '{"road": "机场北", "level": "拥堵", "avg_speed": 35.5}',
    '{"road": "京开", "level": "畅通", "avg_speed": 60.0}',
    '{"road": "机场南", "avg_speed": 30.0}',
    '{"road": "机场北", "level": "严重拥堵", "avg_speed": 25.0}',
    '{"road": "机场北", "level": "缓行",',
]
REQUIRED = ["road", "level", "avg_speed"]
LEVELS = ["畅通", "缓行", "拥堵"]

valid, invalid = [], []
for i, text in enumerate(RAW, 1):
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        print("样本", i, "：JSON 解析失败，已跳过")
        invalid.append(i)
        continue
    missing = [k for k in REQUIRED if k not in data]
    if missing:
        print("样本", i, "：缺字段", missing)
        invalid.append(i)
        continue
    if data["level"] not in LEVELS:
        print("样本", i, "：level 非法", data["level"])
        invalid.append(i)
        continue
    valid.append(data)

print("合格条数：", len(valid))
print("不合格条数：", len(invalid))
print("合格率：", round(len(valid) / len(RAW) * 100, 1), "%")
saved = json.dumps(valid, ensure_ascii=False)
print("落盘 JSON 条数：", len(json.loads(saved)))
print("合格记录：", [(d["road"], d["level"], d["avg_speed"]) for d in valid])

### 运行结果

```text
样本 3 ：缺字段 ['level']
样本 4 ：level 非法 严重拥堵
样本 5 ：JSON 解析失败，已跳过
合格条数： 2
不合格条数： 3
合格率： 40.0 %
落盘 JSON 条数： 2
合格记录： [('机场北', '拥堵', 35.5), ('京开', '畅通', 60.0)]
```

### 解析

1. **修正 1**：三个 `continue` 之前都补 `invalid.append(i)`。注意 `i` 是**样本编号**而不是数据本身，方便事后回到 `RAW` 里复查原始返回——真实项目里这就相当于错误日志的索引。
2. **修正 2**：`len(valid) / len(RAW)` 才是合格率。原写法 `len(valid) / len(valid)` 恒等于 `1.0`，是典型的「分母写错但不会报错」的逻辑 bug，正好用 05A 节学的调试器设断点看变量。
3. **落盘两步走**：`json.dumps(valid, ensure_ascii=False)` 生成字符串，再用 `json.loads(saved)` 回读确认条数。写真实文件时把 `saved` 交给 `Path.write_text(..., encoding="utf-8")` 即可。
4. **不要给坏数据填默认值**：有些学生会用 `data.get("level", "拥堵")` 把缺字段「修好」，这样合格率变成 100%，但标注数据是**伪造**的。数据质量场景中，坏数据必须被记录、被剔除，不能被悄悄改写。
5. **与第 15 章的衔接**：这里的 `valid` 就是 NLP 案例最终写进 CSV / JSON 的那批记录；`invalid` 会写进日志供教师复查提示词。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 不合格条数为 0 | 只在某一分支 append | 三类分支都补 |
| 合格率还是 100% | 分母仍用 `len(valid)` | 改 `len(RAW)` |
| 落盘条数为 5 | 把 `RAW` 整份写进去了 | 只序列化 `valid` |
| 中文变 `\uXXXX` | `dumps` 没加 `ensure_ascii=False` | 补上 |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 8 行输出，不合格 3、合格率 40.0 %、落盘 2 条 | 正常路径 |
| T2 | 样本 4 改为合法等级后合格 3 条，诊断只剩样本 3 与 5 | 分母仍是 5，合格率 60.0 % |
| T3 | 样本 2 缺 `avg_speed`，走缺字段分支，不合格 4 条 | 与取值非法共用记录链 |

---

## 教师用：练习口径问题速查

> 仅供教师备课与批改使用，学生做完再看。

| 练习代码 | 预设问题 | 修复要点 | 参考答案输出 |
|---|---|---|---|
| `practice1_事件文本清洗.py` | ① 用记录条数当路段种类数 ② 低速阈值 `<= 35` | `len(set(roads))`；改 `< 35` | 路段 3、低速车 2 辆 |
| `practice2_标注结果校验.py` | ① 不合格样本未记入 `invalid` ② 分母用合格条数 | 三处 `invalid.append(i)`；`len(RAW)` | 不合格 3、合格率 40.0% |

---

## 本章小结

- **一次本地 AI 调用 = 8 步工程链路**：环境 → 找文件 → 转码 → 发请求 → 收 JSON → 校验 → 清洗 → 落盘，本章逐个补齐，第 15 章把它们串起来。
- **两条主线**：`pathlib` / `base64` / `urllib.request` 负责「数据进出程序」；`json` / `re` / `csv` / `Counter` 负责「数据变形与校验」。
- **三个必须记住的参数**：`json.dumps(..., ensure_ascii=False)`、`urlopen(..., timeout=...)`、`open(..., encoding="utf-8-sig")` 写 CSV。
- **语法约束 ≠ 语义正确**：`format` 只能保证 JSON 可解析，字段齐全与取值合法必须自己校验；坏数据要记录，不要填默认值。
- **离线边界**：首次 `ollama pull` 需联网，之后断网可推理；`127.0.0.1:11434` 只在本机，无账号无密钥。
- **显存决定模型**：8GB 档用 `gemma3:4b` + `qwen2.5:3b`，4GB/6GB 档降到 `moondream` + `qwen2.5:3b`，不碰 11B/13B 视觉模型。
- **统计口径是本练习的核心**：记录数 ≠ 类别数，`<=` ≠ `<`，合格率的分母是样本总数——三者都是「不报错、只算错」，正是 05A 节调试器最该上场的地方。

## 自检清单

- [ ] 能说出 `venv` / `pip` / `requirements.txt` 三者分工，并解释为什么写 `python3 -m pip`；
- [ ] 会用 `Path` 的 `/` 拼接、`glob` 遍历、`suffix.lower()` 过滤，能解释 `PurePath` 与 `Path` 的区别；
- [ ] 能写出 `json.dumps(ensure_ascii=False)` → `.encode("utf-8")` → `json.loads` 的完整往返；
- [ ] 会说明 `format` JSON Schema 的局限，并能独立写出「缺字段 + 取值非法」两层校验；
- [ ] 会用 `base64.b64encode(...).decode("ascii")` 把图片放进 JSON，知道 Ollama 只收纯载荷；
- [ ] 能构造 `Request` 并说出 POST 四要素，知道 `urlopen` 必须带 `timeout` 及两类异常；
- [ ] 会用 `re.search` / `findall` / `sub` 抽取与脱敏，模式一律写 `r"..."`；
- [ ] 会用 `csv.DictReader` / `DictWriter` 与 `Counter.most_common()`，能区分记录数与类别数；
- [ ] 已按 14.9 完成本机部署，填好「本机配置卡」，并做过断网自检；
- [ ] 完成练习 14-1、14-2 的两处口径修正，输出与参考答案逐字一致。

## 前后章节衔接

- 本章把**第 8 章文件操作**从「手工 `split` 读写文本」升级为标准库的 `pathlib` / `csv` / `json`，把**第 10 章模块**从「会 `import`」升级为「会建环境、管依赖」。
- **第 9 章异常处理**在这里第一次承担真实工程职责：`urlopen` 连不上、`json.loads` 解析失败、字段非法，三种失败必须分别兜住并留下记录。
- **05A 节调试技能**在练习 14-1、14-2 直接复用：两道题都是「不报错、只算错」，建议在 `print` 前设断点看 `len(rows)`、`len(valid)`、`len(invalid)` 三个值。
- 学完本章进入**第 15 章本地 AI 综合实战**：案例一「AI 图片标注」与案例二「自然语言处理」将完整跑一遍本章的 8 步链路，用 14.9 填好的「本机配置卡」选模型。
- **终结性考核（99）**仍排在第 15 章之后，考核范围以第 1—13 章为主；第 14—15 章作为本地 AI 拓展模块，供课程设计与学有余力的学生选做。

---

[🏠 课程首页](../00_课程索引.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[◀ 上一章：综合项目实战](../13综合项目实战/13_综合项目实战.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[下一章：本地AI综合实战 ▶](../15本地AI综合实战/15_本地AI综合实战.ipynb)